Setup

In [ ]:
# Install dependencies and Ollama
import subprocess
import time
import threading
import os

# First install zstd (required for Ollama extraction)
print("Installing zstd...")
subprocess.run("apt-get update && apt-get install -y zstd", shell=True, check=True)

# Install Ollama
print("Installing Ollama...")
subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)

# Function to run Ollama server
def run_ollama_server():
    subprocess.run(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# Start Ollama server in a separate thread (not background process)
print("Starting Ollama server...")
server_thread = threading.Thread(target=run_ollama_server, daemon=True)
server_thread.start()

# Wait for server to initialize
time.sleep(10)

# Pull the model (this will take several minutes)
print("Pulling llama3.1:8b (this takes 5-10 minutes)...")
subprocess.run(["ollama", "pull", "llama3.1:8b"], check=True)

print("Ollama ready!")

n

In [ ]:
"""
Single combined prompt — all three aspects extracted in one LLM call.

Extracts three aspects per story, defined as follows (official task definitions):

  Course of Action (CoA)
    The SEQUENCE OF HAPPENINGS in the story — what happens, in what order,
    and what causes what. Described in abstract structural terms: what type
    of events occur and how they chain together. Character names and specific
    locations are replaced with role labels (protagonist, antagonist, ally).

  Outcomes
    The RESULTS OF THE HAPPENINGS, excluding intermediate results that change
    later on. Only the stable final state reached at the end of the story.
    Described in abstract terms: what the protagonist ultimately achieves,
    loses, or becomes.

  Abstract Theme
    The MOTIFS AND THEMES explored in the story. Does NOT cover the concrete
    setting. Universal human experiences, moral questions, and narrative
    archetypes — described without any reference to specific characters,
    locations, or plot events.

WHY THESE DEFINITIONS MATTER FOR SIMILARITY
  Two stories share a CoA when they have the same action structure:
    e.g. "criminal threatens protagonist → protagonist devises plan →
    plan executed → antagonist outwitted" matches across heist films.
  Two stories share an Outcome when they reach the same final state:
    e.g. "protagonist reunites with loved one after long separation"
    matches across very different plot paths.
  Two stories share a Theme when they explore the same human experiences:
    e.g. "loyalty vs institutional duty" matches regardless of genre.

PROMPT DESIGN PRINCIPLES
  1. One-shot example per aspect — shows the model the abstraction level
  2. Explicit negative constraints — forbid character names, specific places
  3. Role-based framing — ask for "protagonist" not character name
  4. Structured output — ask for numbered steps (CoA) or labelled sentences
  5. Length control — tight token budgets prevent over-generation

OUTPUT FORMAT
  aspects_cache_llm_extracted.json — same format as before:
  {"<story_text>": {"coa": "...", "outcomes": "...", "theme": "..."}, ...}
"""

import os, sys, json, time, random, re
from pathlib import Path
from tqdm import tqdm

# Paths
DATA_FILES = {
    "dev_track_a":  "dev_track_a.jsonl",
    "test_track_a": "test_track_a.jsonl",
    "test_track_b": "test_track_b.jsonl",
}

CACHE_PATH   = "/kaggle/working/aspects_cache_llm_extracted3.json"
OLLAMA_MODEL = "llama3.1:8b"
OLLAMA_URL   = "http://localhost:11434/api/generate"

# COMBINED PROMPT
# Single prompt extracting all three aspects in one LLM call.
# Returns a JSON object with keys "coa", "outcomes", "theme".
#
# Design principles:
#   - All three aspect definitions in one context so the model sees
#     how they DIFFER from each other (critical for correct abstraction)
#   - One shared example showing all three aspects for the SAME story
#     (contrast between aspects is clearer than three separate examples)
#   - JSON output enforced via Ollama format parameter (valid JSON guaranteed)
#   - Explicit "having..." prohibition stops outcomes over-generation
#   - Role label requirement for CoA + Theme prevents name leakage
#   - Tight length guidance in output format section

# System message — role and behavioural constraints
SYSTEM_MSG = (
    "You are a precise narrative analyst specialising in story structure. "
    "You follow instructions exactly. "
    "You always output valid JSON with no markdown fences, no extra keys, "
    "and no text outside the JSON object."
)

# Combined extraction prompt — one call, three aspects
COMBINED_PROMPT = """Analyse the story summary and extract three narrative aspects.
Return ONLY valid JSON with keys: "coa", "outcomes", "theme".
No extra text.

━━━ CORE PRINCIPLE ━━━
Each aspect must capture DIFFERENT information:
- COA = process (what happens)
- OUTCOMES = final state (what is true at the end)
- THEME = abstract meaning (what it represents)

Avoid overlap between them.

━━━━━━━━━━━━━━━━━━━━━━━
COA (Course of Action)
━━━━━━━━━━━━━━━━━━━━━━━
Describe the FULL causal sequence of events.

REQUIREMENTS:
- 3-6 numbered steps (1. 2. 3. ...)
- MUST include the FINAL transition into resolution
- Each step = one causal event
- Use abstract action types (escape, betrayal, investigation, confrontation, sacrifice)

USE:
- role labels (protagonist, antagonist, authority, ally)
- generic locations (city, prison, battlefield)

FORBIDDEN:
- character names
- specific places
- themes or emotions
- vague verbs ("deals with", "goes through")

GOOD:
"protagonist investigates → discovers threat → confronts antagonist → resolves conflict"

━━━━━━━━━━━━━━━━━━━━━━━
OUTCOMES (STRICT FORMAT)
━━━━━━━━━━━━━━━━━━━━━━━
Describe ONLY the final stable state.

Write EXACTLY 2 sentences:

Sentence 1:
- protagonist final status (success / failure / survival / transformation)

Sentence 2:
- type of resolution:
  - conflict_resolved / unresolved / partial
  - + nature of change (personal / relational / systemic)

FORBIDDEN:
- "having..." clauses
- process descriptions
- vague words like "things improve"

GOOD:
"Protagonist survives and achieves escape from immediate danger. The conflict is partially resolved with personal transformation but broader threats remain."

━━━━━━━━━━━━━━━━━━━━━━━
THEME (NORMALIZED)
━━━━━━━━━━━━━━━━━━━━━━━
Write 2-4 SHORT phrases (not full sentences).

Each phrase must be:
- abstract
- generalizable across stories

FORMAT:
"theme1; theme2; theme3"

GOOD:
"survival under adversity; loyalty vs duty; human resilience"

BAD:
"The story explores how a man struggles..."

━━━━━━━━━━━━━━━━━━━━━━━
EXAMPLE
━━━━━━━━━━━━━━━━━━━━━━━

Story:
"A young man injures his brother, is placed under supervision, falsely accused, and later proven innocent."

Output:
{
  "coa": "1. Protagonist commits violence and is processed by authority.\n2. Authority imposes supervision and assigns a helper.\n3. Community falsely accuses protagonist, escalating conflict.\n4. Evidence emerges that clears protagonist and resolves accusations.",
  "outcomes": "Protagonist is exonerated and transitions to a stable path of self-improvement. The conflict is resolved with personal transformation and social reintegration.",
  "theme": "redemption; social stigma; justice vs prejudice"
}

━━━━━━━━━━━━━━━━━━━━━━━
NOW ANALYSE
━━━━━━━━━━━━━━━━━━━━━━━

Story: {story}

Output:
"""

# Per-aspect validation thresholds
ASPECT_VALIDATORS = {
    "coa": {
        "min_len": 80,
        "max_len": 700,
        "warn_if_absent": r"^\s*1[\.\)]",
        "warn_msg": "CoA should start with numbered step '1.' or '1)'"
    },
    "outcomes": {
        "min_len": 40,
        "max_len": 300,
        "warn_if_absent": None,
        "warn_msg": ""
    },
    "theme": {
        "min_len": 60,
        "max_len": 450,
        "warn_if_absent": None,
        "warn_msg": ""
    },
}

# STORY COLLECTION

def _norm_key(text):
    return " ".join(str(text).split())


def collect_stories(data_files):
    seen, stories = set(), []

    def _add(text, title, source):
        if text is None:
            return
        key = _norm_key(text)
        if key and key not in seen and len(key) >= 20:
            seen.add(key)
            stories.append({"text": key, "title": title, "source": source})

    for name, path in data_files.items():
        if not Path(path).exists():
            print(f"  [skip] {path} not found")
            continue
        print(f"  Reading {path} ...")
        with open(path, encoding="utf-8") as f:
            for line in f:
                obj = json.loads(line)
                if name == "test_track_b":
                    _add(obj.get("text", ""),
                         obj.get("story_details", {}).get("title", ""), name)
                else:
                    for field, det_key in [
                        ("anchor_text", "story_details_anchor"),
                        ("text_a",      "story_details_a"),
                        ("text_b",      "story_details_b"),
                    ]:
                        _add(obj.get(field, ""),
                             obj.get(det_key, {}).get("title", ""), name)

    print(f"\nUnique stories collected: {len(stories)}")
    return stories


# CACHE

def load_cache(path):
    if Path(path).exists():
        with open(path, encoding="utf-8") as f:
            raw = json.load(f)
        cache = {_norm_key(k): v for k, v in raw.items()}
        print(f"Cache loaded: {len(cache)} entries from {path}")
        return cache
    return {}


def save_cache(cache, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(cache, f, indent=2, ensure_ascii=False)


def is_complete(entry):
    return (isinstance(entry, dict)
            and all(entry.get(k, "").strip() for k in ("coa", "outcomes", "theme")))



# RESPONSE CLEANING & VALIDATION

# Strip preamble echoes from LLM responses (defence-in-depth after JSON parsing)
_PREAMBLE_RE = re.compile(
    r'^(?:'
    # "here is/are [the/a] <any phrase>" followed by newline
    r'here (?:is|are)(?: the| a)?[^\n:]{0,80}?[\s:.-]*\n+'
    r'|'
    # "Certainly!/Sure!" acknowledgements
    r'(?:certainly|sure|of course|absolutely)[!,.]?[^\n]*\n*'
    r'|'
    # Explicit aspect label ("Course of Action:", "Outcomes:", etc.)
    r'(?:course of action|outcomes?|abstract theme|response|answer)\s*[:：]\s*\n*'
    r')',
    re.IGNORECASE
)


def _clean_response(text):
    """Remove preamble echoes and leading labels from LLM responses."""
    if not text:
        return ""
    text = text.strip()
    for _ in range(4):
        cleaned = _PREAMBLE_RE.sub("", text).strip()
        if cleaned == text:
            break
        text = cleaned
    text = re.sub(r"^\s*\n+", "", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def _validate_aspect(aspect_name, text, story_preview):
    """Print a warning if an extracted aspect looks malformed."""
    v = ASPECT_VALIDATORS.get(aspect_name, {})
    if not v:
        return
    issues = []
    if len(text) < v.get("min_len", 0):
        issues.append(f"too short ({len(text)} chars, min={v['min_len']})")
    if len(text) > v.get("max_len", 9999):
        issues.append(f"very long ({len(text)} chars) — may be over-generated")
    pat = v.get("warn_if_absent")
    if pat and not re.search(pat, text, re.MULTILINE):
        issues.append(v.get("warn_msg", ""))
    if aspect_name in ("coa", "theme") and re.search(r'\([A-Z][a-z]+ [A-Z][a-z]+\)', text):
        issues.append("may contain actor name in parentheses")
    if issues:
        print(f"    \u26a0  {aspect_name}: {'; '.join(issues)}")
        print(f"       story: {story_preview[:60]}...")
        print(f"       text:  {text[:100]}")

# OLLAMA BACKEND

def _ollama_available():
    try:
        import urllib.request
        urllib.request.urlopen(
            OLLAMA_URL.replace("/api/generate", ""), timeout=2)
        return True
    except Exception:
        return False


def check_ollama_gpu():
    import urllib.request
    try:
        with urllib.request.urlopen(
            OLLAMA_URL.replace("/api/generate", "/api/ps"), timeout=5
        ) as resp:
            ps = json.loads(resp.read())
        for m in ps.get("models", []):
            vram = m.get("size_vram", 0)
            status = "GPU" if vram > 0 else "CPU ⚠ slow!"
            print(f"  Ollama: {m.get('name')}  VRAM={vram/1e9:.1f}GB  {status}")
    except Exception:
        pass


def _ollama_generate(prompt, max_tokens=500, json_mode=False):
    """
    Call Ollama API.
    json_mode=True adds format="json" to the request — Ollama guarantees
    the response is valid JSON (llama3.1:8b supports this natively).
    """
    import urllib.request
    payload_dict = {
        "model":   OLLAMA_MODEL,
        "prompt":  prompt,
        "stream":  False,
        "options": {
            "num_predict":    max_tokens,
            "temperature":    0.1,   # low — factual extraction
            "top_p":          0.9,
            "repeat_penalty": 1.1,   # discourage prompt echo
        },
    }
    if json_mode:
        payload_dict["format"] = "json"

    payload = json.dumps(payload_dict).encode()
    req = urllib.request.Request(
        OLLAMA_URL, data=payload,
        headers={"Content-Type": "application/json"}, method="POST")
    with urllib.request.urlopen(req, timeout=180) as resp:
        return json.loads(resp.read()).get("response", "").strip()
def _parse_json_response(raw):
    """
    Parse JSON from LLM response. Handles:
      - Clean JSON objects
      - JSON wrapped in markdown fences (```json ... ```)
      - Partial JSON with recoverable structure
    Returns dict with keys "coa", "outcomes", "theme", or raises ValueError.
    """
    text = raw.strip()

    # Strip markdown fences if present
    text = re.sub(r"^```(?:json)?\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\s*```$", "", text)
    text = text.strip()

    # Attempt direct parse
    try:
        obj = json.loads(text)
        if isinstance(obj, dict) and all(k in obj for k in ("coa", "outcomes", "theme")):
            return obj
    except json.JSONDecodeError:
        pass

    # Recovery: extract individual fields with regex if JSON is malformed
    # (handles cases where the model emits valid field values but broken JSON syntax)
    recovered = {}
    for key in ("coa", "outcomes", "theme"):
        # Match: "key": "value" — handles escaped newlines and quotes
        pattern = rf'"{key}"\s*:\s*"((?:[^"\\]|\\.)*)"'
        m = re.search(pattern, text, re.DOTALL)
        if m:
            # Unescape the captured value
            recovered[key] = m.group(1).replace("\\n", "\n").replace('\\"', '"')

    if len(recovered) == 3:
        return recovered

    raise ValueError(f"Could not parse JSON response. Raw: {raw[:200]}")


def extract_with_ollama(story_text, title=""):
    """
    Extract all three narrative aspects in a single LLM call.
    Returns dict with keys: title, coa, outcomes, theme.
    """
    prompt   = SYSTEM_MSG + "\n\n" + COMBINED_PROMPT.format(story=story_text)
    aspects  = {"title": title}

    try:
        raw     = _ollama_generate(prompt, max_tokens=500, json_mode=True)
        parsed  = _parse_json_response(raw)
        aspects["coa"]      = _clean_response(parsed.get("coa",      ""))
        aspects["outcomes"] = _clean_response(parsed.get("outcomes", ""))
        aspects["theme"]    = _clean_response(parsed.get("theme",    ""))
    except Exception as e:
        print(f"    [ollama error] {e}")
        aspects["coa"] = aspects["outcomes"] = aspects["theme"] = ""
        return aspects

    # Validate each extracted aspect
    for asp in ("coa", "outcomes", "theme"):
        _validate_aspect(asp, aspects[asp], story_text)

    return aspects

# CACHE REPAIR

def repair_cache(cache_path):
    """
    Re-run _clean_response on all entries.
    Also flag entries where CoA is missing numbered steps (likely old-format).
    """
    cache = load_cache(cache_path)
    if not cache:
        print("Cache empty.")
        return
    cleaned_fields = 0
    suspicious = []
    for text, entry in cache.items():
        for asp in ("coa", "outcomes", "theme"):
            original = entry.get(asp, "")
            cleaned  = _clean_response(original)
            if cleaned != original:
                entry[asp] = cleaned
                cleaned_fields += 1
        # Flag CoA entries missing numbered structure (old extractions)
        coa = entry.get("coa", "")
        if coa and not re.search(r"^\s*1[\.\)]", coa, re.MULTILINE):
            suspicious.append((text[:60], coa[:80]))

    save_cache(cache, cache_path)
    print(f"Repair complete: {cleaned_fields} fields cleaned, "
          f"{len(suspicious)} CoA entries lack numbered steps.")
    if suspicious:
        print("First 3 suspicious CoA entries:")
        for story, coa in suspicious[:3]:
            print(f"  story: {story}...")
            print(f"  coa:   {coa}...")
    return cache


# QUALITY CHECK

def quality_check(cache, n=10):
    print("\n" + "=" * 70)
    print(f"  QUALITY CHECK — {min(n, len(cache))} random samples")
    print("=" * 70)

    for i, key in enumerate(random.sample(list(cache.keys()),
                                           min(n, len(cache))), 1):
        entry = cache[key]
        print(f"\n[{i}] {entry.get('title','(no title)')}")
        print(f"  Story : {key[:120]}...")
        print(f"  CoA   : {entry.get('coa','(empty)')[:200]}")
        print(f"  Out   : {entry.get('outcomes','(empty)')[:150]}")
        print(f"  Theme : {entry.get('theme','(empty)')[:150]}")
        for asp in ("coa", "outcomes", "theme"):
            _validate_aspect(asp, entry.get(asp, ""), key)

    total    = len(cache)
    complete = sum(1 for v in cache.values() if is_complete(v))
    print(f"\n  Total: {total}  Complete: {complete} ({complete/total*100:.1f}%)"
          if total else "\n  Cache empty.")

    # Check CoA structural quality: fraction with numbered steps
    coa_structured = sum(
        1 for v in cache.values()
        if re.search(r"^\s*1[\.\)]", v.get("coa", ""), re.MULTILINE)
    )
    print(f"  CoA with numbered steps: {coa_structured}/{total} "
          f"({coa_structured/total*100:.1f}%)" if total else "")
    print("=" * 70 + "\n")


# EXTRACTION LOOP

def run_extraction(stories, cache, dry_run=False):
    todo = [s for s in stories if not is_complete(cache.get(s["text"]))]
    if dry_run:
        todo = todo[:5]
        print(f"DRY RUN: processing {len(todo)} stories only\n")
    else:
        print(f"To process: {len(todo)}  "
              f"Already cached: {len(stories) - len(todo)}\n")

    if not todo:
        print("Nothing to do — all stories cached.")
        return cache

    start = time.time()
    errors = 0

    for i, story in enumerate(tqdm(todo, desc="Extracting aspects")):
        try:
            aspects = extract_with_ollama(story["text"], story["title"])
            cache[story["text"]] = aspects
        except Exception as e:
            print(f"\n  [ERROR] story {i}: {e}")
            cache[story["text"]] = {
                "title": story["title"], "coa": "", "outcomes": "", "theme": ""}
            errors += 1

        # Save every 10 stories — crash recovery
        if (i + 1) % 10 == 0 or i == len(todo) - 1:
            save_cache(cache, CACHE_PATH)

        # Progress ETA every 50 stories
        if (i + 1) % 50 == 0:
            elapsed   = time.time() - start
            rate      = (i + 1) / elapsed
            remaining = (len(todo) - i - 1) / rate / 60
            print(f"\n  [{i+1}/{len(todo)}] "
                  f"{rate:.2f} stories/s  ETA {remaining:.1f} min  "
                  f"errors: {errors}")

    elapsed = time.time() - start
    print(f"\nDone in {elapsed/60:.1f} min  |  errors: {errors}")
    return cache


# CONFIG & MAIN

class _Cfg:
    dry_run       = False   # True = process 5 stories only
    quality_check = False   # True = inspect cache and exit
    repair        = False   # True = clean existing cache entries and exit
    cache         = CACHE_PATH
    data_dir      = "/kaggle/input/datasets/similarity"      # empty = use DATA_FILES paths as-is


def main():
    args = _Cfg()

    global CACHE_PATH
    CACHE_PATH = args.cache

    data_files = (
        {name: str(Path(args.data_dir) / fname)
         for name, fname in DATA_FILES.items()}
        if args.data_dir else dict(DATA_FILES)
    )

    print("=" * 60)
    print("  Narrative Aspect Extraction (LLM)")
    print("=" * 60)
    print(f"  Cache   : {CACHE_PATH}")
    print(f"  Backend : Ollama ({OLLAMA_MODEL})")
    print()

    cache = load_cache(CACHE_PATH)

    if args.repair:
        repair_cache(CACHE_PATH)
        return

    if args.quality_check:
        quality_check(cache) if cache else print("Cache empty.")
        return

    if not _ollama_available():
        print("ERROR: Ollama not running. Start with: ollama serve")
        print("       Then pull model: ollama pull llama3.1:8b")
        sys.exit(1)

    check_ollama_gpu()

    print("Collecting stories ...")
    stories = collect_stories(data_files)
    print()

    cache = run_extraction(stories, cache, dry_run=args.dry_run)

    if not args.dry_run:
        quality_check(cache, n=5)

    complete = sum(1 for v in cache.values() if is_complete(v))
    print(f"Final: {len(cache)} entries, {complete} complete")
    print(f"Cache: {CACHE_PATH}")


if __name__ == "__main__":
    main()

In [ ]:
"""
Aspect extraction

Extracts three narrative aspect descriptions per story:
  • Course of Action (CoA)  — what happens, in order
  • Outcomes               — the final resolution
  • Abstract Theme         — universal motifs and patterns

Output: aspects_cache.json
  {
    "<story_text>": {
        "title":    "...",
        "coa":      "...",
        "outcomes": "...",
        "theme":    "..."
    },
    ...
  }
"""

import os, sys, json, time, random, re
from pathlib import Path
from tqdm import tqdm

DATA_FILES = {
    "dev_track_a":  "dev_track_a.jsonl",    # 200 triples
    "test_track_a": "test_track_a.jsonl",   # 400 triples
    "test_track_b": "test_track_b.jsonl",   # 849 individual story texts
}

CACHE_PATH  = "/kaggle/working/aspects_cache.json"
OLLAMA_MODEL  = "llama3.1:8b"
OLLAMA_URL    = "http://localhost:11434/api/generate"

# Prompts

COA_PROMPT = """\
You are a narrative analyst. Read the story summary below and write ONLY \
the sequence of plot events — what happens, in what order, and what causes \
what. Do NOT mention character names, specific locations, or themes. \
Do NOT write any introduction, heading, or label before your answer. \
Begin your response immediately with the first event. Write 2-4 sentences.

Story:
{story}

Response:"""

OUTCOMES_PROMPT = """\
You are a narrative analyst. Read the story summary below and write ONLY \
the final outcome and resolution. What is the end state? What did the \
protagonist ultimately achieve, lose, or experience? Do NOT describe how \
they got there. Do NOT write any introduction, heading, or label. \
Begin your response immediately with the outcome. Write 1-2 sentences.

Story:
{story}

Response:"""

THEME_PROMPT = """\
You are a narrative analyst. Read the story summary below and write ONLY \
the abstract themes and universal human experiences it explores. What \
fundamental aspects of human nature, society, or morality does it examine? \
Do NOT mention specific characters, places, or plot events. \
Do NOT write any introduction, heading, or label. \
Begin your response immediately with the theme. Write 1-3 sentences.

Story:
{story}

Response:"""

PROMPTS = {
    "coa":      COA_PROMPT,
    "outcomes": OUTCOMES_PROMPT,
    "theme":    THEME_PROMPT,
}


# Story collection — deduplicate across all datasets

def collect_stories(data_files):
    """
    Walk all available dataset files and collect every unique story text.
    Returns a list of dicts: [{text, title, source}, ...]
    Deduplication is by exact text string after stripping whitespace.
    """
    seen   = set()
    stories = []

    def _add(text, title, source):
        # Guard against explicit JSON null values
        if text is None:
            return
        key = _normalise_key(text)   # same normalisation as cache keys
        if key and key not in seen and len(key) >= 20:
            seen.add(key)
            stories.append({"text": key, "title": title, "source": source})

    for name, path in data_files.items():
        if not Path(path).exists():
            print(f"  [skip] {path} not found")
            continue
        print(f"  Reading {path} ...")
        with open(path, encoding="utf-8") as f:
            for line in f:
                obj = json.loads(line)

                if name == "test_track_b":
                    # Track B input: one story per row with "text" field
                    _add(obj.get("text",""),
                         obj.get("story_details",{}).get("title",""),
                         name)
                else:
                    # Track A / synthetic: triples with anchor_text, text_a, text_b
                    details = {
                        "anchor_text": obj.get("story_details_anchor",{}).get("title",""),
                        "text_a":      obj.get("story_details_a",{}).get("title",""),
                        "text_b":      obj.get("story_details_b",{}).get("title",""),
                    }
                    for field, title_key in [
                        ("anchor_text", "anchor_text"),
                        ("text_a",      "text_a"),
                        ("text_b",      "text_b"),
                    ]:
                        _add(obj.get(field,""), details[title_key], name)

    print(f"\nTotal unique stories collected: {len(stories)}")
    return stories


# Cache helpers

def _normalise_key(text):
    """Canonical cache key: strip whitespace, normalise internal spaces."""
    return " ".join(str(text).split())

def load_cache(path):
    if Path(path).exists():
        with open(path, encoding="utf-8") as f:
            raw = json.load(f)
        # Re-key with normalised keys so laptop and Kaggle entries match
        cache = {_normalise_key(k): v for k, v in raw.items()}
        print(f"Cache loaded: {len(cache)} entries from {path}")
        return cache
    return {}


def save_cache(cache, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(cache, f, indent=2, ensure_ascii=False)


def is_complete(entry):
    """An entry is complete if all three aspects are non-empty strings."""
    return (
        isinstance(entry, dict) and
        all(entry.get(k,"").strip() for k in ["coa", "outcomes", "theme"])
    )


# Backend: Ollama (local)

def _ollama_available():
    try:
        import urllib.request
        urllib.request.urlopen(OLLAMA_URL.replace("/api/generate", ""), timeout=2)
        return True
    except Exception:
        return False


def check_ollama_gpu():
    """
    Print Ollama GPU status. If Ollama is running on CPU instead of GPU,
    extraction will be ~10x slower. Run this before starting extraction.
    """
    import urllib.request
    try:
        with urllib.request.urlopen(
            OLLAMA_URL.replace("/api/generate", "/api/ps"), timeout=5
        ) as resp:
            ps = json.loads(resp.read())
        models = ps.get("models", [])
        if models:
            for m in models:
                size_vram = m.get("size_vram", 0)
                print(f"  Ollama model: {m.get('name')}  "
                      f"VRAM: {size_vram/1e9:.1f}GB  "
                      f"({'GPU ✓' if size_vram > 0 else 'CPU ⚠ — slow!'})")
        else:
            print("  Ollama running but no model loaded yet (loads on first call)")
    except Exception:
        pass  # ps endpoint may not exist on older Ollama versions


def _ollama_generate(prompt, model=OLLAMA_MODEL, max_tokens=200):
    import urllib.request, urllib.error
    payload = json.dumps({
        "model":  model,
        "prompt": prompt,
        "stream": False,
        "options": {
            "num_predict": max_tokens,
            "temperature": 0.2,   # low temp for factual extraction
            "top_p": 0.9,
        }
    }).encode()

    req = urllib.request.Request(
        OLLAMA_URL,
        data=payload,
        headers={"Content-Type": "application/json"},
        method="POST"
    )
    with urllib.request.urlopen(req, timeout=120) as resp:
        result = json.loads(resp.read())
    return result.get("response", "").strip()



# Response cleaning
# Llama-3.1-8B sometimes echoes a preamble like "Here is the sequence of
# plot events:" before the actual answer, despite explicit prompt instructions.
# This function removes all known preamble patterns and normalises whitespace.

_PREAMBLE_RE = re.compile(
    r'^(?:'
    r'(?:here (?:is|are) (?:the )?(?:a )?'
    r'(?:sequence of plot events|extracted plot events|following|'
    r'abstract themes?.*?|course of action.*?|outcome.*?|theme.*?))'
    r'[:\.!]?\s*\n+'
    r')',
    re.IGNORECASE | re.DOTALL
)

# Also strip leading label like "Course of Action:" or "Response:"
_LABEL_RE = re.compile(
    r'^(?:Course of Action|Outcome(?:s)?|Abstract Theme|Response)\s*[:：]\s*',
    re.IGNORECASE
)

def _clean_response(text):
    """Remove preamble echoes and leading labels from LLM responses."""
    text = text.strip()
    # Iteratively remove preamble lines (sometimes nested)
    for _ in range(3):
        cleaned = _PREAMBLE_RE.sub('', text).strip()
        if cleaned == text:
            break
        text = cleaned
    # Remove any remaining leading label
    text = _LABEL_RE.sub('', text).strip()
    # Normalise internal whitespace — collapse 3+ newlines to 2
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text


def extract_with_ollama(story_text, title=""):
    aspects = {}
    for aspect, prompt_template in PROMPTS.items():
        prompt = prompt_template.format(story=story_text)
        try:
            response = _ollama_generate(prompt)
            # Clean up: remove any accidental repetition of the prompt keyword
            response = _clean_response(response)
            aspects[aspect] = response
        except Exception as e:
            aspects[aspect] = ""
            print(f"    [ollama error] {aspect}: {e}")
    aspects["title"] = title
    return aspects


SYSTEM_MSG = (
    "You are a precise narrative analyst. "
    "Always follow instructions exactly. "
    "Never add introductions, headings, labels, or phrases like "
    "'Here is...' or 'Here are...' before your answer. "
    "Start your response immediately with the requested content."
)


def _build_messages(prompt):
    return [{"role": "system", "content": SYSTEM_MSG},
            {"role": "user",   "content": prompt}]


def _extract_reply(output):
    generated = output["generated_text"]
    if isinstance(generated, list):
        return generated[-1].get("content", "").strip()
    return str(generated).strip()

# Backend auto-detection and dispatch

def detect_backend(forced=None):
    if forced == "ollama":
        if not _ollama_available():
            print("ERROR: Ollama not running. Start it with: ollama serve")
            sys.exit(1)
        return "ollama"

    # Auto-detect: prefer Ollama if running locally
    if _ollama_available():
        print("Backend: Ollama (local) — detected running instance")
        return "ollama"


def extract_aspects(story_text, title, backend):
    if backend == "ollama":
        return extract_with_ollama(story_text, title)
    else:
        return 


# Quality check — print random samples from the cache

def quality_check(cache, n=10):
    print("\n" + "="*70)
    print(f"  QUALITY CHECK — {n} random samples from cache")
    print("="*70)

    sample_keys = random.sample(list(cache.keys()), min(n, len(cache)))
    for i, key in enumerate(sample_keys, 1):
        entry = cache[key]
        print(f"\n[{i}] Title: {entry.get('title', '(unknown)')}")
        print(f"    Story (first 120 chars): {key[:120]}...")
        print(f"    CoA:      {entry.get('coa','(empty)')[:150]}")
        print(f"    Outcomes: {entry.get('outcomes','(empty)')[:150]}")
        print(f"    Theme:    {entry.get('theme','(empty)')[:150]}")
        # Flag potential problems
        for asp in ["coa","outcomes","theme"]:
            val = entry.get(asp,"")
            if len(val) < 20:
                print(f"    ⚠  {asp} looks too short — may need manual fix")
            if any(w in val.lower() for w in ["romeo","juliet","hamlet"]):
                # proper names leaked into extraction
                print(f"    ⚠  {asp} may contain character names")

    print("\n" + "="*70)

    # Summary stats
    total = len(cache)
    complete = sum(1 for v in cache.values() if is_complete(v))
    print(f"  Total entries : {total}")
    print(f"  Complete      : {complete}  ({complete/total*100:.1f}%)")
    print(f"  Incomplete    : {total-complete}")

    # Average lengths
    for asp in ["coa","outcomes","theme"]:
        lengths = [len(v.get(asp,"")) for v in cache.values() if v.get(asp,"")]
        if lengths:
            print(f"  Avg {asp:8s} length: {sum(lengths)/len(lengths):.0f} chars")
    print("="*70 + "\n")


# Main extraction loop

def run_extraction(stories, cache, backend, dry_run=False):
    # Filter to stories not yet in cache (or incomplete)
    todo = [s for s in stories if not is_complete(cache.get(s["text"]))]

    if dry_run:
        todo = todo[:5]
        print(f"DRY RUN: processing {len(todo)} stories only\n")
    else:
        print(f"Stories to process: {len(todo)} "
              f"(already cached: {len(stories)-len(todo)})\n")

    if not todo:
        print("Nothing to do — all stories already cached.")
        return cache

    start_time = time.time()
    errors = 0

    batch_size = 1

    for i in tqdm(range(0, len(todo), batch_size),
                  desc=f"Extracting ({backend})",
                  total=(len(todo) + batch_size - 1) // batch_size):

        batch = todo[i : i + batch_size]

        try:
            for story in batch:
                aspects = extract_aspects(story["text"], story["title"], backend)
                cache[story["text"]] = aspects
        except Exception as e:
            print(f"\n  [ERROR] batch {i}: {e}")
            for story in batch:
                cache[story["text"]] = {
                    "title": story["title"], "coa": "", "outcomes": "", "theme": ""}
            errors += len(batch)

        # Save every 10 batches — crash-safe
        batch_num = i // batch_size
        if batch_num % 10 == 0 or i + batch_size >= len(todo):
            save_cache(cache, CACHE_PATH)

        # Progress estimate every 50 stories
        stories_done = min(i + batch_size, len(todo))
        if stories_done > 0 and stories_done % 50 < batch_size:
            elapsed   = time.time() - start_time
            rate      = stories_done / elapsed
            remaining = (len(todo) - stories_done) / rate / 60
            print(f"\n  [{stories_done}/{len(todo)}] "
                  f"rate={rate:.2f} stories/s  "
                  f"ETA={remaining:.1f} min  "
                  f"errors={errors}")

    save_cache(cache, CACHE_PATH)
    elapsed = time.time() - start_time
    print(f"\nExtraction complete in {elapsed/60:.1f} min  |  errors: {errors}")
    return cache


# ── Cache repair — clean existing cache entries ───────────────────────────────

def repair_cache(cache_path):
    """
    Run _clean_response on every entry in an existing cache file.
    Use this to fix preamble echoes in caches generated with the old prompts.

    Usage:  python extract_aspects_kaggle.py  (set quality_check=False, 
            then call repair_cache(CACHE_PATH) directly)
    """
    cache = load_cache(cache_path)
    if not cache:
        print("Cache is empty or not found.")
        return

    fixed = 0
    for text, entry in cache.items():
        for asp in ["coa", "outcomes", "theme"]:
            original = entry.get(asp, "")
            cleaned  = _clean_response(original)
            if cleaned != original:
                entry[asp] = cleaned
                fixed += 1

    save_cache(cache, cache_path)
    print(f"Cache repair complete — {fixed} fields cleaned across {len(cache)} entries.")
    return cache


class _Cfg:

    backend       = "ollama" 

    # Set True to process only 5 stories — verify output before full run
    dry_run       = False

    # Set True to inspect existing cache and exit without extracting
    quality_check = False

    # Paths
    # Cache output file
    cache         = CACHE_PATH
    data_dir      = "/kaggle/input/datasets/similarity"

    hf_token      = ""   # leave empty

def parse_args():
    return _Cfg()


def main():
    args = parse_args()

    global CACHE_PATH
    CACHE_PATH = args.cache

    # Resolve data file paths
    # If data_dir is empty, DATA_FILES paths are used as-is (relative to cwd)
    if args.data_dir:
        data_files = {
            name: str(Path(args.data_dir) / fname)
            for name, fname in DATA_FILES.items()
        }
    else:
        data_files = dict(DATA_FILES)

    print("=" * 60)
    print("  Narrative Aspect Extraction")
    print("=" * 60)
    print(f"  Cache     : {CACHE_PATH}")
    print(f"  Data dir  : {args.data_dir}")
    print(f"  Backend   : {args.backend}")
    print()

    # Load existing cache
    cache = load_cache(CACHE_PATH)

    # Quality check mode — just inspect existing cache and exit
    if args.quality_check:
        if not cache:
            print("Cache is empty. Run extraction first.")
        else:
            quality_check(cache)
        return

    # Collect all unique stories from all available datasets
    print("Collecting stories from datasets...")
    stories = collect_stories(data_files)
    print()

    # Detect or validate backend
    backend = detect_backend(
        forced=args.backend if args.backend != "auto" else None
    )
    print(f"Using backend: {backend}")
    if backend == "ollama":
        check_ollama_gpu()
    print()

    # Run extraction
    cache = run_extraction(stories, cache, backend, dry_run=args.dry_run)

    # Always show quality check at end of a real (non-dry) run
    if not args.dry_run and cache:
        quality_check(cache, n=5)

    print(f"\nCache saved to: {CACHE_PATH}")
    print(f"Total entries : {len(cache)}")
    complete = sum(1 for v in cache.values() if is_complete(v))
    print(f"Complete      : {complete}/{len(cache)}")


if __name__ == "__main__":
    main()

In [ ]:
"""
Single combined prompt — all three aspects extracted in one LLM call.

Extracts three aspects per story, defined as follows (official task definitions):

  Course of Action (CoA)
    The SEQUENCE OF HAPPENINGS in the story — what happens, in what order,
    and what causes what. Described in abstract structural terms: what type
    of events occur and how they chain together. Character names and specific
    locations are replaced with role labels (protagonist, antagonist, ally).

  Outcomes
    The RESULTS OF THE HAPPENINGS, excluding intermediate results that change
    later on. Only the stable final state reached at the end of the story.
    Described in abstract terms: what the protagonist ultimately achieves,
    loses, or becomes.

  Abstract Theme
    The MOTIFS AND THEMES explored in the story. Does NOT cover the concrete
    setting. Universal human experiences, moral questions, and narrative
    archetypes — described without any reference to specific characters,
    locations, or plot events.

WHY THESE DEFINITIONS MATTER FOR SIMILARITY
  Two stories share a CoA when they have the same action structure:
    e.g. "criminal threatens protagonist → protagonist devises plan →
    plan executed → antagonist outwitted" matches across heist films.
  Two stories share an Outcome when they reach the same final state:
    e.g. "protagonist reunites with loved one after long separation"
    matches across very different plot paths.
  Two stories share a Theme when they explore the same human experiences:
    e.g. "loyalty vs institutional duty" matches regardless of genre.

PROMPT DESIGN PRINCIPLES
  1. One-shot example per aspect — shows the model the abstraction level
  2. Explicit negative constraints — forbid character names, specific places
  3. Role-based framing — ask for "protagonist" not character name
  4. Structured output — ask for numbered steps (CoA) or labelled sentences
  5. Length control — tight token budgets prevent over-generation

OUTPUT FORMAT
  aspects_cache_llm_extracted.json — same format as before:
  {"<story_text>": {"coa": "...", "outcomes": "...", "theme": "..."}, ...}
"""

import os, sys, json, time, random, re
from pathlib import Path
from tqdm import tqdm

# Paths
DATA_FILES = {
    "dev_track_a":  "dev_track_a.jsonl",
    "test_track_a": "test_track_a.jsonl",
    "test_track_b": "test_track_b.jsonl",
}

CACHE_PATH   = "/kaggle/working/aspects_cache_llm_extracted5.json"
OLLAMA_MODEL = "llama3.1:8b"
OLLAMA_URL   = "http://localhost:11434/api/generate"

# COMBINED PROMPT
# Single prompt extracting all three aspects in one LLM call.
# Returns a JSON object with keys "coa", "outcomes", "theme".
#
# Design principles:
#   - All three aspect definitions in one context so the model sees
#     how they DIFFER from each other (critical for correct abstraction)
#   - One shared example showing all three aspects for the SAME story
#     (contrast between aspects is clearer than three separate examples)
#   - JSON output enforced via Ollama format parameter (valid JSON guaranteed)
#   - Explicit "having..." prohibition stops outcomes over-generation
#   - Role label requirement for CoA + Theme prevents name leakage
#   - Tight length guidance in output format section

# System message — role and behavioural constraints
SYSTEM_MSG = (
    "You are a precise narrative analyst specialising in story structure. "
    "You follow instructions exactly. "
    "You always output valid JSON with no markdown fences, no extra keys, "
    "and no text outside the JSON object."
)

# Combined extraction prompt — one call, three aspects
COMBINED_PROMPT = """Analyse the story summary and extract three narrative aspects.
Return ONLY valid JSON with keys: "coa", "outcomes", "theme".
No extra text.

━━━ CORE PRINCIPLE ━━━
Each aspect must capture DIFFERENT information:
- COA = process (what happens)
- OUTCOMES = final state (what is true at the end)
- THEME = abstract meaning (what it represents)

Avoid overlap between them.

━━━━━━━━━━━━━━━━━━━━━━━
COA (Course of Action)
━━━━━━━━━━━━━━━━━━━━━━━
Describe the FULL causal sequence of events.

REQUIREMENTS:
- 3-6 numbered steps (1. 2. 3. ...)
- MUST include the FINAL transition into resolution
- Each step = one causal event
- Use abstract action types (escape, betrayal, investigation, confrontation, sacrifice)

USE:
- role labels (protagonist, antagonist, authority, ally)
- generic locations (city, prison, battlefield)

FORBIDDEN:
- character names
- specific places
- themes or emotions
- vague verbs ("deals with", "goes through")

GOOD:
"protagonist investigates → discovers threat → confronts antagonist → resolves conflict"

━━━━━━━━━━━━━━━━━━━━━━━
OUTCOMES (STRICT FORMAT)
━━━━━━━━━━━━━━━━━━━━━━━
Describe ONLY the final stable state.

Write EXACTLY 2 sentences:

Sentence 1:
- protagonist final status (success / failure / survival / transformation)

Sentence 2:
- type of resolution:
  - conflict_resolved / unresolved / partial
  - + nature of change (personal / relational / systemic)

FORBIDDEN:
- "having..." clauses
- process descriptions
- vague words like "things improve"

GOOD:
"Protagonist survives and achieves escape from immediate danger. The conflict is partially resolved with personal transformation but broader threats remain."

━━━━━━━━━━━━━━━━━━━━━━━
THEME (NORMALIZED)
━━━━━━━━━━━━━━━━━━━━━━━
Write 2-4 SHORT phrases (not full sentences).

Each phrase must be:
- abstract
- generalizable across stories

FORMAT:
"theme1; theme2; theme3"

GOOD:
"survival under adversity; loyalty vs duty; human resilience"

BAD:
"The story explores how a man struggles..."

━━━━━━━━━━━━━━━━━━━━━━━
EXAMPLE
━━━━━━━━━━━━━━━━━━━━━━━

Story:
"A young man injures his brother, is placed under supervision, falsely accused, and later proven innocent."

Output:
{{
  "coa": "1. Protagonist commits violence and is processed by authority.\n2. Authority imposes supervision and assigns a helper.\n3. Community falsely accuses protagonist, escalating conflict.\n4. Evidence emerges that clears protagonist and resolves accusations.",
  "outcomes": "Protagonist is exonerated and transitions to a stable path of self-improvement. The conflict is resolved with personal transformation and social reintegration.",
  "theme": "redemption; social stigma; justice vs prejudice"
}}

━━━━━━━━━━━━━━━━━━━━━━━
NOW ANALYSE
━━━━━━━━━━━━━━━━━━━━━━━

Story: {story}

Output:
"""

# Per-aspect validation thresholds
ASPECT_VALIDATORS = {
    "coa": {
        "min_len": 80,
        "max_len": 700,
        "warn_if_absent": r"^\s*1[\.\)]",
        "warn_msg": "CoA should start with numbered step '1.' or '1)'"
    },
    "outcomes": {
        "min_len": 40,
        "max_len": 300,
        "warn_if_absent": None,
        "warn_msg": ""
    },
    "theme": {
        "min_len": 60,
        "max_len": 450,
        "warn_if_absent": None,
        "warn_msg": ""
    },
}

# STORY COLLECTION

def _norm_key(text):
    return " ".join(str(text).split())


def collect_stories(data_files):
    seen, stories = set(), []

    def _add(text, title, source):
        if text is None:
            return
        key = _norm_key(text)
        if key and key not in seen and len(key) >= 20:
            seen.add(key)
            stories.append({"text": key, "title": title, "source": source})

    for name, path in data_files.items():
        if not Path(path).exists():
            print(f"  [skip] {path} not found")
            continue
        print(f"  Reading {path} ...")
        with open(path, encoding="utf-8") as f:
            for line in f:
                obj = json.loads(line)
                if name == "test_track_b":
                    _add(obj.get("text", ""),
                         obj.get("story_details", {}).get("title", ""), name)
                else:
                    for field, det_key in [
                        ("anchor_text", "story_details_anchor"),
                        ("text_a",      "story_details_a"),
                        ("text_b",      "story_details_b"),
                    ]:
                        _add(obj.get(field, ""),
                             obj.get(det_key, {}).get("title", ""), name)

    print(f"\nUnique stories collected: {len(stories)}")
    return stories


# CACHE

def load_cache(path):
    if Path(path).exists():
        with open(path, encoding="utf-8") as f:
            raw = json.load(f)
        cache = {_norm_key(k): v for k, v in raw.items()}
        print(f"Cache loaded: {len(cache)} entries from {path}")
        return cache
    return {}


def save_cache(cache, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(cache, f, indent=2, ensure_ascii=False)


def is_complete(entry):
    return (isinstance(entry, dict)
            and all(entry.get(k, "").strip() for k in ("coa", "outcomes", "theme")))



# RESPONSE CLEANING & VALIDATION

# Strip preamble echoes from LLM responses (defence-in-depth after JSON parsing)
_PREAMBLE_RE = re.compile(
    r'^(?:'
    # "here is/are [the/a] <any phrase>" followed by newline
    r'here (?:is|are)(?: the| a)?[^\n:]{0,80}?[\s:.-]*\n+'
    r'|'
    # "Certainly!/Sure!" acknowledgements
    r'(?:certainly|sure|of course|absolutely)[!,.]?[^\n]*\n*'
    r'|'
    # Explicit aspect label ("Course of Action:", "Outcomes:", etc.)
    r'(?:course of action|outcomes?|abstract theme|response|answer)\s*[:：]\s*\n*'
    r')',
    re.IGNORECASE
)


def _clean_response(text):
    """Remove preamble echoes and leading labels from LLM responses."""
    if not text:
        return ""
    text = text.strip()
    for _ in range(4):
        cleaned = _PREAMBLE_RE.sub("", text).strip()
        if cleaned == text:
            break
        text = cleaned
    text = re.sub(r"^\s*\n+", "", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def _validate_aspect(aspect_name, text, story_preview):
    """Print a warning if an extracted aspect looks malformed."""
    v = ASPECT_VALIDATORS.get(aspect_name, {})
    if not v:
        return
    issues = []
    if len(text) < v.get("min_len", 0):
        issues.append(f"too short ({len(text)} chars, min={v['min_len']})")
    if len(text) > v.get("max_len", 9999):
        issues.append(f"very long ({len(text)} chars) — may be over-generated")
    pat = v.get("warn_if_absent")
    if pat and not re.search(pat, text, re.MULTILINE):
        issues.append(v.get("warn_msg", ""))
    if aspect_name in ("coa", "theme") and re.search(r'\([A-Z][a-z]+ [A-Z][a-z]+\)', text):
        issues.append("may contain actor name in parentheses")
    if issues:
        print(f"    \u26a0  {aspect_name}: {'; '.join(issues)}")
        print(f"       story: {story_preview[:60]}...")
        print(f"       text:  {text[:100]}")


# OLLAMA BACKEND

def _ollama_available():
    try:
        import urllib.request
        urllib.request.urlopen(
            OLLAMA_URL.replace("/api/generate", ""), timeout=2)
        return True
    except Exception:
        return False


def check_ollama_gpu():
    import urllib.request
    try:
        with urllib.request.urlopen(
            OLLAMA_URL.replace("/api/generate", "/api/ps"), timeout=5
        ) as resp:
            ps = json.loads(resp.read())
        for m in ps.get("models", []):
            vram = m.get("size_vram", 0)
            status = "GPU ✓" if vram > 0 else "CPU ⚠ slow!"
            print(f"  Ollama: {m.get('name')}  VRAM={vram/1e9:.1f}GB  {status}")
    except Exception:
        pass


def _ollama_generate(prompt, max_tokens=500, json_mode=False):
    """
    Call Ollama API.
    json_mode=True adds format="json" to the request — Ollama guarantees
    the response is valid JSON (llama3.1:8b supports this natively).
    """
    import urllib.request
    payload_dict = {
        "model":   OLLAMA_MODEL,
        "prompt":  prompt,
        "stream":  False,
        "options": {
            "num_predict":    max_tokens,
            "temperature":    0.1,   # low — factual extraction
            "top_p":          0.9,
            "repeat_penalty": 1.1,   # discourage prompt echo
        },
    }
    if json_mode:
        payload_dict["format"] = "json"

    payload = json.dumps(payload_dict).encode()
    req = urllib.request.Request(
        OLLAMA_URL, data=payload,
        headers={"Content-Type": "application/json"}, method="POST")
    with urllib.request.urlopen(req, timeout=180) as resp:
        return json.loads(resp.read()).get("response", "").strip()
def _parse_json_response(raw):
    """
    Parse JSON from LLM response. Handles:
      - Clean JSON objects
      - JSON wrapped in markdown fences (```json ... ```)
      - Partial JSON with recoverable structure
    Returns dict with keys "coa", "outcomes", "theme", or raises ValueError.
    """
    text = raw.strip()

    # Strip markdown fences if present
    text = re.sub(r"^```(?:json)?\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\s*```$", "", text)
    text = text.strip()

    # Attempt direct parse
    try:
        obj = json.loads(text)
        if isinstance(obj, dict) and all(k in obj for k in ("coa", "outcomes", "theme")):
            return obj
    except json.JSONDecodeError:
        pass

    # Recovery: extract individual fields with regex if JSON is malformed
    # (handles cases where the model emits valid field values but broken JSON syntax)
    recovered = {}
    for key in ("coa", "outcomes", "theme"):
        # Match: "key": "value" — handles escaped newlines and quotes
        pattern = rf'"{key}"\s*:\s*"((?:[^"\\]|\\.)*)"'
        m = re.search(pattern, text, re.DOTALL)
        if m:
            # Unescape the captured value
            recovered[key] = m.group(1).replace("\\n", "\n").replace('\\"', '"')

    if len(recovered) == 3:
        return recovered

    raise ValueError(f"Could not parse JSON response. Raw: {raw[:200]}")


def extract_with_ollama(story_text, title=""):
    """
    Extract all three narrative aspects in a single LLM call.
    Returns dict with keys: title, coa, outcomes, theme.
    """
    prompt   = SYSTEM_MSG + "\n\n" + COMBINED_PROMPT.format(story=story_text)
    aspects  = {"title": title}

    try:
        raw     = _ollama_generate(prompt, max_tokens=500, json_mode=True)
        parsed  = _parse_json_response(raw)
        aspects["coa"]      = _clean_response(parsed.get("coa",      ""))
        aspects["outcomes"] = _clean_response(parsed.get("outcomes", ""))
        aspects["theme"]    = _clean_response(parsed.get("theme",    ""))
    except Exception as e:
        print(f"    [ollama error] {e}")
        aspects["coa"] = aspects["outcomes"] = aspects["theme"] = ""
        return aspects

    # Validate each extracted aspect
    for asp in ("coa", "outcomes", "theme"):
        _validate_aspect(asp, aspects[asp], story_text)

    return aspects

# CACHE REPAIR

def repair_cache(cache_path):
    """
    Re-run _clean_response on all entries.
    Also flag entries where CoA is missing numbered steps (likely old-format).
    """
    cache = load_cache(cache_path)
    if not cache:
        print("Cache empty.")
        return
    cleaned_fields = 0
    suspicious = []
    for text, entry in cache.items():
        for asp in ("coa", "outcomes", "theme"):
            original = entry.get(asp, "")
            cleaned  = _clean_response(original)
            if cleaned != original:
                entry[asp] = cleaned
                cleaned_fields += 1
        # Flag CoA entries missing numbered structure (old extractions)
        coa = entry.get("coa", "")
        if coa and not re.search(r"^\s*1[\.\)]", coa, re.MULTILINE):
            suspicious.append((text[:60], coa[:80]))

    save_cache(cache, cache_path)
    print(f"Repair complete: {cleaned_fields} fields cleaned, "
          f"{len(suspicious)} CoA entries lack numbered steps.")
    if suspicious:
        print("First 3 suspicious CoA entries:")
        for story, coa in suspicious[:3]:
            print(f"  story: {story}...")
            print(f"  coa:   {coa}...")
    return cache


# QUALITY CHECK

def quality_check(cache, n=10):
    print("\n" + "=" * 70)
    print(f"  QUALITY CHECK — {min(n, len(cache))} random samples")
    print("=" * 70)

    for i, key in enumerate(random.sample(list(cache.keys()),
                                           min(n, len(cache))), 1):
        entry = cache[key]
        print(f"\n[{i}] {entry.get('title','(no title)')}")
        print(f"  Story : {key[:120]}...")
        print(f"  CoA   : {entry.get('coa','(empty)')[:200]}")
        print(f"  Out   : {entry.get('outcomes','(empty)')[:150]}")
        print(f"  Theme : {entry.get('theme','(empty)')[:150]}")
        for asp in ("coa", "outcomes", "theme"):
            _validate_aspect(asp, entry.get(asp, ""), key)

    total    = len(cache)
    complete = sum(1 for v in cache.values() if is_complete(v))
    print(f"\n  Total: {total}  Complete: {complete} ({complete/total*100:.1f}%)"
          if total else "\n  Cache empty.")

    # Check CoA structural quality: fraction with numbered steps
    coa_structured = sum(
        1 for v in cache.values()
        if re.search(r"^\s*1[\.\)]", v.get("coa", ""), re.MULTILINE)
    )
    print(f"  CoA with numbered steps: {coa_structured}/{total} "
          f"({coa_structured/total*100:.1f}%)" if total else "")
    print("=" * 70 + "\n")


# EXTRACTION LOOP

def run_extraction(stories, cache, dry_run=False):
    todo = [s for s in stories if not is_complete(cache.get(s["text"]))]
    if dry_run:
        todo = todo[:5]
        print(f"DRY RUN: processing {len(todo)} stories only\n")
    else:
        print(f"To process: {len(todo)}  "
              f"Already cached: {len(stories) - len(todo)}\n")

    if not todo:
        print("Nothing to do — all stories cached.")
        return cache

    start = time.time()
    errors = 0

    for i, story in enumerate(tqdm(todo, desc="Extracting aspects")):
        try:
            aspects = extract_with_ollama(story["text"], story["title"])
            cache[story["text"]] = aspects
        except Exception as e:
            print(f"\n  [ERROR] story {i}: {e}")
            cache[story["text"]] = {
                "title": story["title"], "coa": "", "outcomes": "", "theme": ""}
            errors += 1

        # Save every 10 stories — crash recovery
        if (i + 1) % 10 == 0 or i == len(todo) - 1:
            save_cache(cache, CACHE_PATH)

        # Progress ETA every 50 stories
        if (i + 1) % 50 == 0:
            elapsed   = time.time() - start
            rate      = (i + 1) / elapsed
            remaining = (len(todo) - i - 1) / rate / 60
            print(f"\n  [{i+1}/{len(todo)}] "
                  f"{rate:.2f} stories/s  ETA {remaining:.1f} min  "
                  f"errors: {errors}")

    elapsed = time.time() - start
    print(f"\nDone in {elapsed/60:.1f} min  |  errors: {errors}")
    return cache


# CONFIG & MAIN

class _Cfg:
    dry_run       = False   # True = process 5 stories only
    quality_check = False   # True = inspect cache and exit
    repair        = False   # True = clean existing cache entries and exit
    cache         = CACHE_PATH
    data_dir      = "/kaggle/input/datasets/similarity"      # empty = use DATA_FILES paths as-is


def main():
    args = _Cfg()

    global CACHE_PATH
    CACHE_PATH = args.cache

    data_files = (
        {name: str(Path(args.data_dir) / fname)
         for name, fname in DATA_FILES.items()}
        if args.data_dir else dict(DATA_FILES)
    )

    print("=" * 60)
    print("  Narrative Aspect Extraction (LLM)")
    print("=" * 60)
    print(f"  Cache   : {CACHE_PATH}")
    print(f"  Backend : Ollama ({OLLAMA_MODEL})")
    print()

    cache = load_cache(CACHE_PATH)

    if args.repair:
        repair_cache(CACHE_PATH)
        return

    if args.quality_check:
        quality_check(cache) if cache else print("Cache empty.")
        return

    if not _ollama_available():
        print("ERROR: Ollama not running. Start with: ollama serve")
        print("       Then pull model: ollama pull llama3.1:8b")
        sys.exit(1)

    check_ollama_gpu()

    print("Collecting stories ...")
    stories = collect_stories(data_files)
    print()

    cache = run_extraction(stories, cache, dry_run=args.dry_run)

    if not args.dry_run:
        quality_check(cache, n=5)

    complete = sum(1 for v in cache.values() if is_complete(v))
    print(f"Final: {len(cache)} entries, {complete} complete")
    print(f"Cache: {CACHE_PATH}")


if __name__ == "__main__":
    main()


Improve aspect extraction

In [ ]:
"""
Single combined prompt — all three aspects extracted in one LLM call.

Extracts three aspects per story, defined as follows (official task definitions):

  Course of Action (CoA)
    The SEQUENCE OF HAPPENINGS in the story — what happens, in what order,
    and what causes what. Described in abstract structural terms: what type
    of events occur and how they chain together. Character names and specific
    locations are replaced with role labels (protagonist, antagonist, ally).

  Outcomes
    The RESULTS OF THE HAPPENINGS, excluding intermediate results that change
    later on. Only the stable final state reached at the end of the story.
    Described in abstract terms: what the protagonist ultimately achieves,
    loses, or becomes.

  Abstract Theme
    The MOTIFS AND THEMES explored in the story. Does NOT cover the concrete
    setting. Universal human experiences, moral questions, and narrative
    archetypes — described without any reference to specific characters,
    locations, or plot events.

WHY THESE DEFINITIONS MATTER FOR SIMILARITY
  Two stories share a CoA when they have the same action structure:
    e.g. "criminal threatens protagonist → protagonist devises plan →
    plan executed → antagonist outwitted" matches across heist films.
  Two stories share an Outcome when they reach the same final state:
    e.g. "protagonist reunites with loved one after long separation"
    matches across very different plot paths.
  Two stories share a Theme when they explore the same human experiences:
    e.g. "loyalty vs institutional duty" matches regardless of genre.

PROMPT DESIGN PRINCIPLES
  1. One-shot example per aspect — shows the model the abstraction level
  2. Explicit negative constraints — forbid character names, specific places
  3. Role-based framing — ask for "protagonist" not character name
  4. Structured output — ask for short structural clauses (CoA) and literal end-state sentences
  5. Length control — tight token budgets prevent over-generation

OUTPUT FORMAT
  aspects_cache_llm_extracted.json — same format as before:
  {"<story_text>": {"coa": "...", "outcomes": "...", "theme": "..."}, ...}
"""

import os, sys, json, time, random, re
from pathlib import Path
from tqdm import tqdm

# Paths
DATA_FILES = {
    "dev_track_a":  "dev_track_a.jsonl",
    "test_track_a": "test_track_a.jsonl",
    "test_track_b": "test_track_b.jsonl",
}

CACHE_PATH   = "/kaggle/working/aspects_cache_llm_extracted_improved.json"
OLLAMA_MODEL = "llama3.1:8b"
OLLAMA_URL   = "http://localhost:11434/api/generate"

# COMBINED PROMPT
# Single prompt extracting all three aspects in one LLM call.
# Returns a JSON object with keys "coa", "outcomes", "theme".
#
# Design principles:
#   - All three aspect definitions in one context so the model sees
#     how they DIFFER from each other (critical for correct abstraction)
#   - One shared example showing all three aspects for the SAME story
#     (contrast between aspects is clearer than three separate examples)
#   - JSON output enforced via Ollama format parameter (valid JSON guaranteed)
#   - Explicit "having..." prohibition stops outcomes over-generation
#   - Role label requirement for CoA + Theme prevents name leakage
#   - Tight length guidance in output format section

# System message — role and behavioural constraints
SYSTEM_MSG = (
    "You are a precise narrative analyst specialising in story structure. "
    "You follow instructions exactly. "
    "You always output valid JSON with no markdown fences, no extra keys, "
    "and no text outside the JSON object. "
    "You never invent events, outcomes, or successful resolutions that are not clearly supported by the story."
)

# Combined extraction prompt — one call, three aspects
COMBINED_PROMPT = """Analyse the story summary and extract three narrative aspects.
Return ONLY valid JSON with keys: "coa", "outcomes", "theme".
No extra text.

━━━ CORE PRINCIPLE ━━━
Each aspect must capture DIFFERENT information:
- COA = process (what happens)
- OUTCOMES = final state (what is true at the end)
- THEME = abstract meaning (what it represents)

Avoid overlap between them.

━━━━━━━━━━━━━━━━━━━━━━━
COA (Course of Action)
━━━━━━━━━━━━━━━━━━━━━━━
Describe the FULL causal sequence of events.

REQUIREMENTS:
- 3-5 short clauses separated by " ; "
- MUST include the FINAL transition into resolution
- Each clause = one causal event
- Keep clauses short and structural, not long plot-summary prose
- Do not use numbering, bullet points, or arrows
- Use abstract action types (escape, betrayal, investigation, confrontation, sacrifice)

USE:
- role labels (protagonist, antagonist, authority, ally)
- generic locations (city, prison, battlefield)

FORBIDDEN:
- character names
- specific places
- themes or emotions
- vague verbs ("deals with", "goes through")

GOOD:
"protagonist investigates → discovers threat → confronts antagonist → resolves conflict"

━━━━━━━━━━━━━━━━━━━━━━━
OUTCOMES (STRICT FORMAT)
━━━━━━━━━━━━━━━━━━━━━━━
Describe ONLY the final stable state explicitly supported by the summary.

Write EXACTLY 2 sentences:

Sentence 1:
- state the clearest end-state for the protagonist or central conflict
- use cautious wording if the ending is unclear

Sentence 2:
- state exactly one resolution label:
  - conflict resolved
  - conflict unresolved
  - conflict partially resolved

CRITICAL:
- Do NOT infer success, justice served, systemic change, broader implications, or future consequences unless clearly stated.
- If the ending is unclear, say so explicitly.
- Prefer literal wording over interpretation.

FORBIDDEN:
- "having..." clauses
- process descriptions
- vague words like "things improve"
- phrases like "broader threats remain", "personal transformation", "systemic change", "justice served", "new hope" unless explicitly stated

GOOD:
"Protagonist escapes the immediate danger, but the broader situation remains uncertain. conflict partially resolved."

━━━━━━━━━━━━━━━━━━━━━━━
THEME (NORMALIZED)
━━━━━━━━━━━━━━━━━━━━━━━
Write 2-4 SHORT phrases (not full sentences).

Each phrase must be:
- abstract
- generalizable across stories
- 1-5 words long
- lower-case concept phrase

FORMAT:
"theme1; theme2; theme3"

GOOD:
"survival under adversity; loyalty vs duty; human resilience"

BAD:
"The story explores how a man struggles..."

━━━━━━━━━━━━━━━━━━━━━━━
EXAMPLE
━━━━━━━━━━━━━━━━━━━━━━━

Story:
"A young man injures his brother, is placed under supervision, falsely accused, and later proven innocent."

Output:
{{
  "coa": "protagonist commits violence; authority imposes supervision; community escalates accusations; evidence clears protagonist",
  "outcomes": "Protagonist is exonerated and allowed to move forward. conflict resolved.",
  "theme": "redemption; social stigma; justice vs prejudice"
}}

━━━━━━━━━━━━━━━━━━━━━━━
NOW ANALYSE
━━━━━━━━━━━━━━━━━━━━━━━

Story: {story}

Output:
"""

REPAIR_PROMPT = """You are repairing a noisy narrative-aspect extraction.
Return ONLY valid JSON with keys: "coa", "outcomes", "theme".

Requirements:
- remove character names and specific places
- keep coa as 3-5 short structural clauses separated by " ; "
- remove numbering, arrows, and bullet points from coa
- keep outcomes as exactly 2 short cautious sentences about final state only
- make the second outcomes sentence exactly one of: "conflict resolved." / "conflict unresolved." / "conflict partially resolved."
- keep theme as 2-4 short lower-case phrases separated by semicolons
- do not invent success, justice served, systemic change, broader implications, or details not clearly present in the story

Story:
{story}

Current extraction:
{bad_json}

Return repaired JSON only.
"""

# Per-aspect validation thresholds
ASPECT_VALIDATORS = {
    "coa": {
        "min_len": 80,
        "max_len": 700,
        "warn_if_absent": None,
        "warn_msg": ""
    },
    "outcomes": {
        "min_len": 35,
        "max_len": 220,
        "warn_if_absent": None,
        "warn_msg": ""
    },
    "theme": {
        "min_len": 12,
        "max_len": 120,
        "warn_if_absent": None,
        "warn_msg": ""
    },
}

# STORY COLLECTION

def _norm_key(text):
    return " ".join(str(text).split())


def collect_stories(data_files):
    seen, stories = set(), []

    def _add(text, title, source):
        if text is None:
            return
        key = _norm_key(text)
        if key and key not in seen and len(key) >= 20:
            seen.add(key)
            stories.append({"text": key, "title": title, "source": source})

    for name, path in data_files.items():
        if not Path(path).exists():
            print(f"  [skip] {path} not found")
            continue
        print(f"  Reading {path} ...")
        with open(path, encoding="utf-8") as f:
            for line in f:
                obj = json.loads(line)
                if name == "test_track_b":
                    _add(obj.get("text", ""),
                         obj.get("story_details", {}).get("title", ""), name)
                else:
                    for field, det_key in [
                        ("anchor_text", "story_details_anchor"),
                        ("text_a",      "story_details_a"),
                        ("text_b",      "story_details_b"),
                    ]:
                        _add(obj.get(field, ""),
                             obj.get(det_key, {}).get("title", ""), name)

    print(f"\nUnique stories collected: {len(stories)}")
    return stories


# CACHE

def load_cache(path):
    if Path(path).exists():
        with open(path, encoding="utf-8") as f:
            raw = json.load(f)
        cache = {_norm_key(k): v for k, v in raw.items()}
        print(f"Cache loaded: {len(cache)} entries from {path}")
        return cache
    return {}


def save_cache(cache, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(cache, f, indent=2, ensure_ascii=False)


def is_complete(entry):
    return (isinstance(entry, dict)
            and all(entry.get(k, "").strip() for k in ("coa", "outcomes", "theme")))



# RESPONSE CLEANING & VALIDATION

# Strip preamble echoes from LLM responses (defence-in-depth after JSON parsing)
_PREAMBLE_RE = re.compile(
    r'^(?:'
    # "here is/are [the/a] <any phrase>" followed by newline
    r'here (?:is|are)(?: the| a)?[^\n:]{0,80}?[\s:.-]*\n+'
    r'|'
    # "Certainly!/Sure!" acknowledgements
    r'(?:certainly|sure|of course|absolutely)[!,.]?[^\n]*\n*'
    r'|'
    # Explicit aspect label ("Course of Action:", "Outcomes:", etc.)
    r'(?:course of action|outcomes?|abstract theme|response|answer)\s*[:：]\s*\n*'
    r')',
    re.IGNORECASE
)


def _clean_response(text):
    """Remove preamble echoes and leading labels from LLM responses."""
    if not text:
        return ""
    text = text.strip()
    for _ in range(4):
        cleaned = _PREAMBLE_RE.sub("", text).strip()
        if cleaned == text:
            break
        text = cleaned
    text = re.sub(r"^\s*\n+", "", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def _theme_parts(text):
    return [p.strip(" -.;,").lower() for p in re.split(r"[;\n,]", text) if p.strip(" -.;,")]


def _postprocess_coa(text):
    text = _clean_response(text)
    text = re.sub(r"\s*\n\s*", " ", text)
    text = re.sub(r"(?m)^\s*(\d+)[\.\)]\s*", "", text)
    text = re.sub(r"(?i)\bstep\s+\d+\s*[:.-]\s*", "", text)
    text = re.sub(r"\s*[→\-]+\s*", "; ", text)
    text = re.sub(r"\s*;\s*", "; ", text)
    text = re.sub(r"(?i)\b(protagonist|antagonist|authority|ally)\s*:\s*", r"\1 ", text)
    text = re.sub(r"\s{2,}", " ", text)
    text = text.strip(" ;")
    return text.strip()


def _postprocess_outcomes(text):
    text = _clean_response(text)
    text = re.sub(r"\bresolved with partial resolution\b", "partially resolved", text, flags=re.IGNORECASE)
    text = re.sub(r"\bconflict is resolved with unresolved\b", "conflict remains unresolved", text, flags=re.IGNORECASE)
    text = re.sub(r"\bprotagonist succeeds in\b", "protagonist attempts to", text, flags=re.IGNORECASE)
    text = re.sub(r"\bultimately (solves|resolves)\b", "ultimately addresses", text, flags=re.IGNORECASE)
    text = re.sub(r"\bfully resolves\b", "largely resolves", text, flags=re.IGNORECASE)
    text = re.sub(r"\b(personal transformation|systemic change|justice served|new hope|broader (threats|issues|relationships) remain)\b", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def _postprocess_coa(text):
    text = _clean_response(text)
    text = re.sub(r"\s*\n\s*", " ", text)
    text = re.sub(r"(?m)^\s*(\d+)[\.\)]\s*", "", text)
    text = re.sub(r"(?i)\bstep\s+\d+\s*[:.-]\s*", "", text)
    text = re.sub(r"\s*(?:->|→)\s*", "; ", text)
    text = re.sub(r"\s-\s", "; ", text)
    text = re.sub(r"\s*;\s*", "; ", text)
    text = re.sub(r"(?i)\b(protagonist|antagonist|authority|ally)\s*:\s*", r"\1 ", text)
    text = re.sub(r"\s{2,}", " ", text)
    return text.strip(" ;")


def _postprocess_theme(text):
    phrases = []
    seen = set()
    for part in _theme_parts(_clean_response(text)):
        part = re.sub(r"\b(the|a|an|story explores|themes? of)\b", "", part, flags=re.IGNORECASE)
        part = re.sub(r"\b(moral of the story|central theme|main theme)\b", "", part, flags=re.IGNORECASE)
        part = re.sub(r"\s+", " ", part).strip(" -.;,")
        if not part:
            continue
        if len(part.split()) > 5:
            continue
        if part not in seen:
            seen.add(part)
            phrases.append(part)
    return "; ".join(phrases[:4])


def _aspect_issues(aspect_name, text):
    """Return a list of issues if an extracted aspect looks malformed."""
    v = ASPECT_VALIDATORS.get(aspect_name, {})
    issues = []
    if not v:
        return issues
    if len(text) < v.get("min_len", 0):
        issues.append(f"too short ({len(text)} chars, min={v['min_len']})")
    if len(text) > v.get("max_len", 9999):
        issues.append(f"very long ({len(text)} chars) - may be over-generated")
    pat = v.get("warn_if_absent")
    if pat and not re.search(pat, text, re.MULTILINE):
        issues.append(v.get("warn_msg", ""))
    if aspect_name in ("coa", "theme") and re.search(r'\([A-Z][a-z]+ [A-Z][a-z]+\)', text):
        issues.append("may contain actor name in parentheses")
    if aspect_name == "coa":
        if re.search(r"(^|[\s;])\d+[\.\)]", text):
            issues.append("coa should not contain numbering")
        if re.search(r"\b(Mary Murphy|Antarctica|Earth|Sacramento)\b", text):
            issues.append("may contain specific names or places")
        if "->" in text or "→" in text:
            issues.append("coa should use semicolon-separated clauses, not arrows")
    if aspect_name == "outcomes" and re.search(
        r"\b(succeeds in|ultimately solves|fully resolves|stabilizing markets?|justice served|systemic change|personal transformation|new hope|broader (threats|issues|relationships) remain)\b",
        text,
        re.IGNORECASE,
    ):
        issues.append("may be overconfident or infer unsupported consequences")
    if aspect_name == "outcomes":
        sentences = [s.strip() for s in re.split(r"(?<=[.!?])\s+", text) if s.strip()]
        if len(sentences) != 2:
            issues.append("outcomes should contain exactly 2 short sentences")
        elif sentences[1].lower() not in {
            "conflict resolved.",
            "conflict unresolved.",
            "conflict partially resolved.",
        }:
            issues.append("second outcomes sentence should be a plain resolution label")
    if aspect_name == "theme":
        parts = _theme_parts(text)
        if len(parts) < 2:
            issues.append("should contain 2-4 semicolon-separated themes")
        if any(len(p.split()) > 5 for p in parts):
            issues.append("theme phrases should be short")
    return issues


def _needs_repair(aspects):
    return any(_aspect_issues(name, aspects.get(name, "")) for name in ("coa", "outcomes", "theme"))


def _legacy_validate_aspect(aspect_name, text, story_preview):
    """Print a warning if an extracted aspect looks malformed."""
    v = ASPECT_VALIDATORS.get(aspect_name, {})
    if not v:
        return
    issues = []
    if len(text) < v.get("min_len", 0):
        issues.append(f"too short ({len(text)} chars, min={v['min_len']})")
    if len(text) > v.get("max_len", 9999):
        issues.append(f"very long ({len(text)} chars) — may be over-generated")
    pat = v.get("warn_if_absent")
    if pat and not re.search(pat, text, re.MULTILINE):
        issues.append(v.get("warn_msg", ""))
    if aspect_name in ("coa", "theme") and re.search(r'\([A-Z][a-z]+ [A-Z][a-z]+\)', text):
        issues.append("may contain actor name in parentheses")
    if issues:
        print(f"    \u26a0  {aspect_name}: {'; '.join(issues)}")
        print(f"       story: {story_preview[:60]}...")
        print(f"       text:  {text[:100]}")


def _validate_aspect(aspect_name, text, story_preview):
    """Print a warning if an extracted aspect looks malformed."""
    issues = _aspect_issues(aspect_name, text)
    if issues:
        print(f"    \u26a0  {aspect_name}: {'; '.join(issues)}")
        print(f"       story: {story_preview[:60]}...")
        print(f"       text:  {text[:100]}")


# OLLAMA BACKEND

def _ollama_available():
    try:
        import urllib.request
        urllib.request.urlopen(
            OLLAMA_URL.replace("/api/generate", ""), timeout=2)
        return True
    except Exception:
        return False


def check_ollama_gpu():
    import urllib.request
    try:
        with urllib.request.urlopen(
            OLLAMA_URL.replace("/api/generate", "/api/ps"), timeout=5
        ) as resp:
            ps = json.loads(resp.read())
        for m in ps.get("models", []):
            vram = m.get("size_vram", 0)
            status = "GPU ✓" if vram > 0 else "CPU ⚠ slow!"
            print(f"  Ollama: {m.get('name')}  VRAM={vram/1e9:.1f}GB  {status}")
    except Exception:
        pass


def _ollama_generate(prompt, max_tokens=500, json_mode=False):
    """
    Call Ollama API.
    json_mode=True adds format="json" to the request — Ollama guarantees
    the response is valid JSON (llama3.1:8b supports this natively).
    """
    import urllib.request
    payload_dict = {
        "model":   OLLAMA_MODEL,
        "prompt":  prompt,
        "stream":  False,
        "options": {
            "num_predict":    max_tokens,
            "temperature":    0.0,   # deterministic, conservative extraction
            "top_p":          0.9,
            "repeat_penalty": 1.1,   # discourage prompt echo
        },
    }
    if json_mode:
        payload_dict["format"] = "json"

    payload = json.dumps(payload_dict).encode()
    req = urllib.request.Request(
        OLLAMA_URL, data=payload,
        headers={"Content-Type": "application/json"}, method="POST")
    with urllib.request.urlopen(req, timeout=180) as resp:
        return json.loads(resp.read()).get("response", "").strip()
def _parse_json_response(raw):
    """
    Parse JSON from LLM response. Handles:
      - Clean JSON objects
      - JSON wrapped in markdown fences (```json ... ```)
      - Partial JSON with recoverable structure
    Returns dict with keys "coa", "outcomes", "theme", or raises ValueError.
    """
    text = raw.strip()

    # Strip markdown fences if present
    text = re.sub(r"^```(?:json)?\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\s*```$", "", text)
    text = text.strip()

    # Attempt direct parse
    try:
        obj = json.loads(text)
        if isinstance(obj, dict) and all(k in obj for k in ("coa", "outcomes", "theme")):
            return obj
    except json.JSONDecodeError:
        pass

    # Recovery: extract individual fields with regex if JSON is malformed
    # (handles cases where the model emits valid field values but broken JSON syntax)
    recovered = {}
    for key in ("coa", "outcomes", "theme"):
        # Match: "key": "value" — handles escaped newlines and quotes
        pattern = rf'"{key}"\s*:\s*"((?:[^"\\]|\\.)*)"'
        m = re.search(pattern, text, re.DOTALL)
        if m:
            # Unescape the captured value
            recovered[key] = m.group(1).replace("\\n", "\n").replace('\\"', '"')

    if len(recovered) == 3:
        return recovered

    raise ValueError(f"Could not parse JSON response. Raw: {raw[:200]}")


def extract_with_ollama(story_text, title=""):
    """
    Extract all three narrative aspects in a single LLM call.
    Returns dict with keys: title, coa, outcomes, theme.
    """
    prompt   = SYSTEM_MSG + "\n\n" + COMBINED_PROMPT.format(story=story_text)
    aspects  = {"title": title}

    try:
        raw     = _ollama_generate(prompt, max_tokens=500, json_mode=True)
        parsed  = _parse_json_response(raw)
        aspects["coa"]      = _postprocess_coa(parsed.get("coa", ""))
        aspects["outcomes"] = _postprocess_outcomes(parsed.get("outcomes", ""))
        aspects["theme"]    = _postprocess_theme(parsed.get("theme", ""))
    except Exception as e:
        print(f"    [ollama error] {e}")
        aspects["coa"] = aspects["outcomes"] = aspects["theme"] = ""
        return aspects

    if _needs_repair(aspects):
        try:
            repair_prompt = REPAIR_PROMPT.format(
                story=story_text,
                bad_json=json.dumps(
                    {"coa": aspects["coa"], "outcomes": aspects["outcomes"], "theme": aspects["theme"]},
                    ensure_ascii=False,
                ),
            )
            raw_repair = _ollama_generate(SYSTEM_MSG + "\n\n" + repair_prompt, max_tokens=350, json_mode=True)
            repaired = _parse_json_response(raw_repair)
            aspects["coa"] = _postprocess_coa(repaired.get("coa", aspects["coa"]))
            aspects["outcomes"] = _postprocess_outcomes(repaired.get("outcomes", aspects["outcomes"]))
            aspects["theme"] = _postprocess_theme(repaired.get("theme", aspects["theme"]))
        except Exception as e:
            print(f"    [repair warning] {e}")

    # Validate each extracted aspect
    for asp in ("coa", "outcomes", "theme"):
        _validate_aspect(asp, aspects[asp], story_text)

    return aspects

# CACHE REPAIR

def repair_cache(cache_path):
    """
    Re-run _clean_response on all entries.
    Also flag entries where CoA still contains numbered steps or arrows.
    """
    cache = load_cache(cache_path)
    if not cache:
        print("Cache empty.")
        return
    cleaned_fields = 0
    suspicious = []
    for text, entry in cache.items():
        processors = {
            "coa": _postprocess_coa,
            "outcomes": _postprocess_outcomes,
            "theme": _postprocess_theme,
        }
        for asp in ("coa", "outcomes", "theme"):
            original = entry.get(asp, "")
            cleaned = processors[asp](original)
            if cleaned != original:
                entry[asp] = cleaned
                cleaned_fields += 1
        # Flag CoA entries still using old structure
        coa = entry.get("coa", "")
        if coa and (re.search(r"(^|[\s;])\d+[\.\)]", coa) or "->" in coa or "→" in coa):
            suspicious.append((text[:60], coa[:80]))

    save_cache(cache, cache_path)
    print(f"Repair complete: {cleaned_fields} fields cleaned, "
          f"{len(suspicious)} CoA entries still look like old-format output.")
    if suspicious:
        print("First 3 suspicious CoA entries:")
        for story, coa in suspicious[:3]:
            print(f"  story: {story}...")
            print(f"  coa:   {coa}...")
    return cache


# QUALITY CHECK

def quality_check(cache, n=10):
    print("\n" + "=" * 70)
    print(f"  QUALITY CHECK — {min(n, len(cache))} random samples")
    print("=" * 70)

    for i, key in enumerate(random.sample(list(cache.keys()),
                                           min(n, len(cache))), 1):
        entry = cache[key]
        print(f"\n[{i}] {entry.get('title','(no title)')}")
        print(f"  Story : {key[:120]}...")
        print(f"  CoA   : {entry.get('coa','(empty)')[:200]}")
        print(f"  Out   : {entry.get('outcomes','(empty)')[:150]}")
        print(f"  Theme : {entry.get('theme','(empty)')[:150]}")
        for asp in ("coa", "outcomes", "theme"):
            _validate_aspect(asp, entry.get(asp, ""), key)

    total    = len(cache)
    complete = sum(1 for v in cache.values() if is_complete(v))
    print(f"\n  Total: {total}  Complete: {complete} ({complete/total*100:.1f}%)"
          if total else "\n  Cache empty.")

    # Check CoA structural quality: fraction already converted to clause format
    coa_structured = sum(
        1 for v in cache.values()
        if ";" in v.get("coa", "") and not re.search(r"(^|[\s;])\d+[\.\)]", v.get("coa", ""))
    )
    print(f"  CoA in semicolon-clause format: {coa_structured}/{total} "
          f"({coa_structured/total*100:.1f}%)" if total else "")
    print("=" * 70 + "\n")


# EXTRACTION LOOP

def run_extraction(stories, cache, dry_run=False):
    todo = [s for s in stories if not is_complete(cache.get(s["text"]))]
    if dry_run:
        todo = todo[:5]
        print(f"DRY RUN: processing {len(todo)} stories only\n")
    else:
        print(f"To process: {len(todo)}  "
              f"Already cached: {len(stories) - len(todo)}\n")

    if not todo:
        print("Nothing to do — all stories cached.")
        return cache

    start = time.time()
    errors = 0

    for i, story in enumerate(tqdm(todo, desc="Extracting aspects")):
        try:
            aspects = extract_with_ollama(story["text"], story["title"])
            cache[story["text"]] = aspects
        except Exception as e:
            print(f"\n  [ERROR] story {i}: {e}")
            cache[story["text"]] = {
                "title": story["title"], "coa": "", "outcomes": "", "theme": ""}
            errors += 1

        # Save every 10 stories — crash recovery
        if (i + 1) % 10 == 0 or i == len(todo) - 1:
            save_cache(cache, CACHE_PATH)

        # Progress ETA every 50 stories
        if (i + 1) % 50 == 0:
            elapsed   = time.time() - start
            rate      = (i + 1) / elapsed
            remaining = (len(todo) - i - 1) / rate / 60
            print(f"\n  [{i+1}/{len(todo)}] "
                  f"{rate:.2f} stories/s  ETA {remaining:.1f} min  "
                  f"errors: {errors}")

    elapsed = time.time() - start
    print(f"\nDone in {elapsed/60:.1f} min  |  errors: {errors}")
    return cache


# CONFIG & MAIN

class _Cfg:
    dry_run       = False   # True = process 5 stories only
    quality_check = False   # True = inspect cache and exit
    repair        = False   # True = clean existing cache entries and exit
    cache         = CACHE_PATH
    data_dir      = "/kaggle/input/datasets/similarity"      # empty = use DATA_FILES paths as-is


def main():
    args = _Cfg()

    global CACHE_PATH
    CACHE_PATH = args.cache

    data_files = (
        {name: str(Path(args.data_dir) / fname)
         for name, fname in DATA_FILES.items()}
        if args.data_dir else dict(DATA_FILES)
    )

    print("=" * 60)
    print("  Narrative Aspect Extraction (LLM)")
    print("=" * 60)
    print(f"  Cache   : {CACHE_PATH}")
    print(f"  Backend : Ollama ({OLLAMA_MODEL})")
    print()

    cache = load_cache(CACHE_PATH)

    if args.repair:
        repair_cache(CACHE_PATH)
        return

    if args.quality_check:
        quality_check(cache) if cache else print("Cache empty.")
        return

    if not _ollama_available():
        print("ERROR: Ollama not running. Start with: ollama serve")
        print("       Then pull model: ollama pull llama3.1:8b")
        sys.exit(1)

    check_ollama_gpu()

    print("Collecting stories ...")
    stories = collect_stories(data_files)
    print()

    cache = run_extraction(stories, cache, dry_run=args.dry_run)

    if not args.dry_run:
        quality_check(cache, n=5)

    complete = sum(1 for v in cache.values() if is_complete(v))
    print(f"Final: {len(cache)} entries, {complete} complete")
    print(f"Cache: {CACHE_PATH}")


if __name__ == "__main__":
    main()


Generate more synthetic data

In [ ]:
import requests
import json
import time
import random
from tqdm import tqdm

# CONFIG
OLLAMA_URL = "http://localhost:11434/api/generate"
MODEL = "llama3.1:8b"
NUM_SAMPLES = 500  
OUTPUT_FILE = "synthetic_stories_llama.jsonl"

# PROMPTS

SEED_PROMPT = """Write a short story of 5–8 sentences in the style
of a Wikipedia film plot summary. Do not include a title, year, or meta information.
Describe only the plot in a neutral, encyclopedic tone, as if summarizing the events of a film."""

SIMILAR_PROMPT = """Write a new story that closely follows the same context and general storyline.
Keep the same sequence of events and plot structure, but change all names of people, places,
and specific details of the setting. The result should feel like a different story while clearly
mirroring the original plot. Write it in 5–8 sentences, in the style of a Wikipedia film plot summary.
Do not include a title, year, or meta information—output the plot only.

Original story:
{story}
"""

DISTANT_PROMPT = """Write a new short story of 5–8 sentences that is only loosely inspired by the original.
You may change the setting, characters, genre, tone, and sequence of events. You are encouraged to create a
different conflict and resolution, and the ending should not mirror the source. The story should retain only a faint
thematic connection or mood from the original, while otherwise standing as a distinctly new narrative.
Write it in the style of a Wikipedia film plot summary. Do not include a title, year, or meta information—output the plot only.

Original story:
{story}
"""

# OLLAMA CALL

def ollama_generate(prompt, temperature=0.7, max_retries=3):
    payload = {
        "model": MODEL,
        "prompt": prompt,
        "stream": False,
        "options": {
            "temperature": temperature,
            "top_p": 0.9,
            "repeat_penalty": 1.1
        }
    }

    for attempt in range(max_retries):
        try:
            response = requests.post(OLLAMA_URL, json=payload, timeout=60)
            response.raise_for_status()
            data = response.json()
            return data.get("response", "").strip()
        except Exception as e:
            print(f"Retry {attempt+1} failed: {e}")
            time.sleep(2)

    return ""


# CLEANING 

def clean_text(text):
    return text.replace("\n", " ").strip()


# GENERATION LOOP

def generate_dataset(n_samples):
    dataset = []

    for _ in tqdm(range(n_samples)):
        # 1. Seed story
        anchor = clean_text(ollama_generate(SEED_PROMPT))

        if not anchor:
            continue

        # 2. Similar story (text_a)
        text_a = clean_text(
            ollama_generate(SIMILAR_PROMPT.format(story=anchor))
        )

        # 3. Distant story (text_b)
        text_b = clean_text(
            ollama_generate(DISTANT_PROMPT.format(story=anchor))
        )

        if not text_a or not text_b:
            continue

        sample = {
            "model": MODEL,
            "anchor_text": anchor,
            "text_a": text_a,
            "text_b": text_b,
            "text_a_is_closer": True
        }

        dataset.append(sample)

    return dataset


# SAVE

def save_jsonl(data, filename):
    with open(filename, "w", encoding="utf-8") as f:
        for item in data:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")



if __name__ == "__main__":
    data = generate_dataset(NUM_SAMPLES)
    save_jsonl(data, OUTPUT_FILE)
    print(f"Saved {len(data)} samples to {OUTPUT_FILE}")

In [ ]:
"""
Generates three types of triples, each designed to target a specific
weakness in the training signal:

  Type 1 — Standard   (organizer-style)
    Anchor + Similar (same CoA/Outcomes, different surface) + Distant
    Covers the easy end of the difficulty spectrum.

  Type 2 — Aspect-Controlled
    Precisely vary ONE aspect at a time while holding others constant.
    Four sub-patterns:
      (a) same_coa_diff_outcome  — trains Outcomes heads
      (b) diff_coa_same_outcome  — trains CoA heads
      (c) same_coa_same_outcome  — unambiguous positive (easy)
      (d) diff_coa_diff_outcome  — hard negative with shared theme only

  Type 3 — Hard Negatives
    Both candidates share surface features (genre, setting, character
    types) with the anchor but differ in plot structure.
    Targets the 74 "hard samples" (62.72% system accuracy in the paper)
    where the difficult cases cost most models the most points.

Why these three types?
  • The organizer synthetic data (1900 triples) uses Type 1 only.
    All positives are "clearly mirroring the original plot", which is
    easier than the actual test set (LLM-rejection-sampled hard cases).
  • Aspect-Controlled triples give the aspect-projection heads a direct
    supervised signal for which aspect dimension matters per triple.
  • Hard negatives teach the model to look past surface similarity
    (same genre, same character archetypes) to narrative structure.

Output format (JSONL):
  {
    "model":           "claude-sonnet-4 | llama3.1:8b | ...",
    "anchor_text":     "...",
    "text_a":          "...",
    "text_b":          "...",
    "text_a_is_closer": true | false,
    "generation_type": "standard | aspect_coa | aspect_outcome | hard_negative",
    "seed_genre":      "...",
    "seed_archetype":  "..."
  }

Usage (Kaggle / Colab with Ollama):
    python build_synthetic_data.py

The script is crash-safe: it writes to JSONL incrementally and resumes
from the last completed triple on restart.
"""

import json
import os
import random
import re
import sys
import time
import urllib.request
from pathlib import Path
from typing import Optional

# CONFIG - DIRECT VALUES (no command line args)

OUTPUT_PATH      = "/kaggle/working/synthetic_data_new.jsonl"
CACHE_PATH       = "/kaggle/working/synthetic_gen_cache.json"   # crash recovery

# Backend configuration - using Ollama
BACKEND          = "ollama"  # "ollama" or "api"
OLLAMA_MODEL     = "llama3.1:8b"
OLLAMA_URL       = "http://localhost:11434/api/generate"
API_MODEL        = "claude-sonnet-4-20250514"   # Anthropic API (not used with ollama)

# How many triples to generate per type (total = 1900)
N_STANDARD       = 700
N_ASPECT         = 900   # 225 per sub-pattern (a/b/c/d)
N_HARD_NEG       = 300
N_TOTAL          = N_STANDARD + N_ASPECT + N_HARD_NEG  # = 1900

SEED             = 42

# SEED TOPICS

GENRES = [
    "crime thriller", "romantic comedy", "war drama", "science fiction",
    "heist film", "political drama", "coming-of-age", "psychological thriller",
    "adventure film", "supernatural horror", "legal drama", "espionage thriller",
    "road movie", "survival story", "family drama", "sports drama",
    "historical epic", "detective mystery", "redemption story", "revenge thriller",
    "courtroom drama", "dystopian fiction", "western", "medical drama",
    "conspiracy thriller",
]

ARCHETYPES = [
    "a protagonist seeks revenge for a loved one's death",
    "an outsider must prove themselves worthy to join a group",
    "a character discovers a dark secret about their own family",
    "two enemies are forced to cooperate in order to survive",
    "a protagonist must choose between duty and personal loyalty",
    "an underdog competes against a powerful corrupt establishment",
    "a character tries to escape and leave behind their past identity",
    "a mentor and protégé relationship is shattered by betrayal",
    "an investigator slowly realizes they are part of the conspiracy they are uncovering",
    "a protagonist sacrifices everything for the sake of others",
    "a false accusation forces a character to prove their innocence",
    "a character is gradually manipulated and begins to lose touch with reality",
    "strangers from opposing worlds form an unlikely bond during a shared crisis",
    "an obsessive pursuit of a single goal ultimately destroys the pursuer",
    "a character returns home after a long absence to find everything has changed",
    "a person of privilege confronts the consequences of their past actions",
    "two people compete for the same goal but fall in love along the way",
    "a whistleblower exposes a powerful institution at great personal cost",
    "a reformed criminal is pulled back into their old life by circumstances",
    "a character must complete one final mission before they can be free",
]

# PROMPTS

# System message — shared across all prompts
SYSTEM = (
    "You are a creative writing assistant specialising in Wikipedia-style "
    "film plot summaries. You write in a neutral, encyclopedic third-person "
    "tone. You never include titles, years, or any meta-information — only "
    "the plot. Each summary is 5–8 sentences long."
)

# ── Type 1: Standard (organizer-style) ───────────────────────────────────────

SEED_PROMPT = """\
Write a short story of 5–8 sentences in the style of a Wikipedia film plot \
summary. The story should be in the genre of {genre} and feature the \
following central situation: {archetype}.

Do not include a title, year, or meta information. Describe only the plot \
in a neutral, encyclopedic tone, as if summarising the events of a film.
Begin your response immediately with the story. No preamble."""

SIMILAR_PROMPT = """\
Here is a story summary:

{anchor}

Write a new story that closely follows the same general situation and plot \
structure. Keep the same sequence of events and outcome, but change all \
character names, specific locations, and surface details of the setting. \
The result should feel like a clearly different story while mirroring the \
original narrative structure.

Write it in 5–8 sentences, in the style of a Wikipedia film plot summary. \
Do not include a title, year, or meta information — output the plot only. \
Begin immediately with the story."""

DISTANT_PROMPT = """\
Here is a story summary:

{anchor}

Write a new short story of 5–8 sentences that is only loosely inspired by \
the original. You may change the setting, characters, genre, tone, and \
sequence of events. Create a different central conflict and a different \
resolution. The ending should not mirror the source. The story should \
retain only a faint thematic connection or mood from the original, while \
otherwise standing as a distinctly new narrative.

Write it in the style of a Wikipedia film plot summary. Do not include a \
title, year, or meta information — output the plot only. \
Begin immediately with the story."""

# ── Type 2: Aspect-Controlled ─────────────────────────────────────────────────

# (a) Same CoA, different Outcome
SAME_COA_DIFF_OUTCOME = """\
Here is a story summary:

{anchor}

Write a new story that follows EXACTLY the same sequence of plot events \
as the original — the same type of actions occur in the same order, and \
the same causal chain connects them. However, the FINAL OUTCOME must be \
clearly different: the protagonist should end up in a meaningfully \
different state (succeeds vs. fails, survives vs. dies, is reunited vs. \
remains separated, etc.).

Change all character names, specific locations, and surface details. \
Keep the abstract plot structure identical but change only the ending.

Write 5–8 sentences. Wikipedia film plot style. No title, year, or meta \
information. Begin immediately with the story."""

# (b) Different CoA, same Outcome
DIFF_COA_SAME_OUTCOME = """\
Here is a story summary:

{anchor}

Write a new story where the protagonist ends up in EXACTLY the same final \
state as in the original (the same outcome: achieves the same goal, \
suffers the same loss, reaches the same resolution). However, the PATH \
they take to get there must be completely different — different types of \
events, different obstacles, different causal chain.

Change all character names, specific locations, and surface details. \
Keep only the final outcome the same.

Write 5–8 sentences. Wikipedia film plot style. No title, year, or meta \
information. Begin immediately with the story."""

# (c) Same CoA AND same Outcome (unambiguous similar)
SAME_COA_SAME_OUTCOME = """\
Here is a story summary:

{anchor}

Write a new story that is narratively very close to the original: the \
same sequence of events occurs in the same order, and the protagonist \
reaches the same final outcome. This should feel like a clear retelling \
of the same story structure.

Change ALL character names, locations, and surface details so it reads \
as a completely different story, but keep the plot structure and ending \
identical.

Write 5–8 sentences. Wikipedia film plot style. No title, year, or meta \
information. Begin immediately with the story."""

# (d) Different CoA AND different Outcome — hard negative (same theme only)
DIFF_COA_DIFF_OUTCOME = """\
Here is a story summary:

{anchor}

Write a new story that explores the SAME ABSTRACT THEMES and human \
experiences as the original, but has a completely different sequence of \
events AND a different final outcome. The surface situation (genre, \
setting, character types) should feel loosely familiar, but the plot \
structure and ending should be entirely different.

This is a HARD NEGATIVE: the stories should share thematic resonance but \
differ in narrative structure.

Change all names and specific details. Write 5–8 sentences. Wikipedia \
film plot style. No title, year, or meta information. Begin immediately."""

# ── Type 3: Hard Negative ──────────────────────────────────────────────────────

HARD_NEGATIVE_SIMILAR_PROMPT = """\
Here is a story summary:

{anchor}

Write a new story that shares the same genre, the same type of \
protagonist (by role, not name), and the same general setting category \
(e.g., both are war stories, both are courtroom dramas). The new story \
should feel superficially similar on first reading.

However, the PLOT STRUCTURE must be genuinely different: different \
sequence of events, different central conflict, and a clearly different \
outcome. This is a hard negative — similar surface, different narrative \
structure.

Write 5–8 sentences. Wikipedia film plot style. No title, year, or meta \
information. Begin immediately with the story."""

HARD_NEGATIVE_DISTANT_PROMPT = """\
Here is a story summary:

{anchor}

Write a completely different story in a different genre, with a different \
type of protagonist, different setting, and different plot structure. \
The only connection to the original should be a very faint thematic mood.

Write 5–8 sentences. Wikipedia film plot style. No title, year, or meta \
information. Begin immediately with the story."""

# GENERATION BACKEND

def _ollama_available():
    try:
        urllib.request.urlopen(
            OLLAMA_URL.replace("/api/generate", ""), timeout=2)
        return True
    except Exception:
        return False


def _anthropic_available():
    return bool(os.environ.get("ANTHROPIC_API_KEY", ""))


def _generate_ollama(prompt: str, max_tokens: int = 300) -> str:
    payload = json.dumps({
        "model":   OLLAMA_MODEL,
        "prompt":  SYSTEM + "\n\n" + prompt,
        "stream":  False,
        "options": {
            "num_predict":    max_tokens,
            "temperature":    0.8,
            "top_p":          0.95,
            "repeat_penalty": 1.1,
        },
    }).encode()
    req = urllib.request.Request(
        OLLAMA_URL, data=payload,
        headers={"Content-Type": "application/json"}, method="POST")
    with urllib.request.urlopen(req, timeout=120) as resp:
        return json.loads(resp.read()).get("response", "").strip()


def _generate_api(prompt: str, max_tokens: int = 300) -> str:
    """Call the Anthropic /v1/messages API."""
    api_key = os.environ.get("ANTHROPIC_API_KEY", "")
    if not api_key:
        raise RuntimeError("ANTHROPIC_API_KEY not set")

    payload = json.dumps({
        "model":      API_MODEL,
        "max_tokens": max_tokens,
        "system":     SYSTEM,
        "messages":   [{"role": "user", "content": prompt}],
    }).encode()

    req = urllib.request.Request(
        "https://api.anthropic.com/v1/messages",
        data=payload,
        headers={
            "Content-Type":         "application/json",
            "x-api-key":            api_key,
            "anthropic-version":    "2023-06-01",
        },
        method="POST",
    )
    with urllib.request.urlopen(req, timeout=60) as resp:
        data = json.loads(resp.read())
    return data["content"][0]["text"].strip()


def generate(prompt: str, max_tokens: int = 300) -> str:
    """Generate text with retry logic using configured backend."""
    for attempt in range(3):
        try:
            if BACKEND == "ollama":
                return _generate_ollama(prompt, max_tokens)
            else:
                return _generate_api(prompt, max_tokens)
        except Exception as e:
            if attempt == 2:
                print(f"  [ERROR] Generation failed after 3 attempts: {e}")
                return ""
            wait = 2 ** attempt
            print(f"  [retry {attempt+1}] {e}  — waiting {wait}s")
            time.sleep(wait)
    return ""


def _clean(text: str) -> str:
    """Remove common LLM preambles and leading labels."""
    text = text.strip()
    # "Here is the story: <content>" / "Here's a summary:\n<content>" / "Sure! ..."
    # Matches preamble up to first colon or newline, strips the preamble
    text = re.sub(
        r"^(?:here(?:'s| is)|sure[!,]?|certainly[!,]?)"
        r"(?:[^:\n]*)(?::\s*|\n+)",
        '', text, flags=re.IGNORECASE).strip()
    # **Title:** or **Story:** markdown headers
    text = re.sub(r'^\*\*[^*]+\*\*\s*\n+', '', text)
    # "Story:" or "Summary:" standalone label at start
    text = re.sub(r'^(?:story|summary|plot)\s*:\s*', '', text,
                  flags=re.IGNORECASE)
    return text.strip()


# TRIPLE BUILDERS

def _is_valid(text: str, min_words: int = 20) -> bool:
    return bool(text) and len(text.split()) >= min_words


def build_standard_triple(genre: str, archetype: str,
                           model_name: str) -> Optional[dict]:
    """Type 1: anchor + similar + distant, 50/50 label swap."""
    anchor = _clean(generate(
        SEED_PROMPT.format(genre=genre, archetype=archetype)))
    if not _is_valid(anchor):
        return None

    similar = _clean(generate(SIMILAR_PROMPT.format(anchor=anchor)))
    distant = _clean(generate(DISTANT_PROMPT.format(anchor=anchor)))

    if not _is_valid(similar) or not _is_valid(distant):
        return None

    # 50/50 label swap to prevent positional bias
    if random.random() < 0.5:
        text_a, text_b, closer = similar, distant, True
    else:
        text_a, text_b, closer = distant, similar, False

    return {
        "model":            model_name,
        "anchor_text":      anchor,
        "text_a":           text_a,
        "text_b":           text_b,
        "text_a_is_closer": closer,
        "generation_type":  "standard",
        "seed_genre":       genre,
        "seed_archetype":   archetype,
    }


def build_aspect_triple(genre: str, archetype: str, pattern: str,
                         model_name: str) -> Optional[dict]:
    """
    Type 2: aspect-controlled.
    pattern ∈ {same_coa_diff_outcome, diff_coa_same_outcome,
               same_coa_same_outcome, diff_coa_diff_outcome}
    """
    anchor = _clean(generate(
        SEED_PROMPT.format(genre=genre, archetype=archetype)))
    if not _is_valid(anchor):
        return None

    prompt_map = {
        "same_coa_diff_outcome": SAME_COA_DIFF_OUTCOME,
        "diff_coa_same_outcome": DIFF_COA_SAME_OUTCOME,
        "same_coa_same_outcome": SAME_COA_SAME_OUTCOME,
        "diff_coa_diff_outcome": DIFF_COA_DIFF_OUTCOME,
    }

    # For d: diff_coa_diff_outcome → the aspect-variant IS the distant story
    # and we need a clearly similar story as the other candidate
    if pattern in ("same_coa_diff_outcome", "diff_coa_same_outcome",
                   "same_coa_same_outcome"):
        # aspect_variant is the SIMILAR story
        aspect_variant = _clean(
            generate(prompt_map[pattern].format(anchor=anchor)))
        distant = _clean(generate(DISTANT_PROMPT.format(anchor=anchor)))

        if not _is_valid(aspect_variant) or not _is_valid(distant):
            return None

        if random.random() < 0.5:
            text_a, text_b, closer = aspect_variant, distant, True
        else:
            text_a, text_b, closer = distant, aspect_variant, False

    else:  # diff_coa_diff_outcome → hard negative case
        hard_neg = _clean(
            generate(prompt_map[pattern].format(anchor=anchor)))
        similar = _clean(generate(SIMILAR_PROMPT.format(anchor=anchor)))

        if not _is_valid(hard_neg) or not _is_valid(similar):
            return None

        if random.random() < 0.5:
            text_a, text_b, closer = similar, hard_neg, True
        else:
            text_a, text_b, closer = hard_neg, similar, False

    return {
        "model":            model_name,
        "anchor_text":      anchor,
        "text_a":           text_a,
        "text_b":           text_b,
        "text_a_is_closer": closer,
        "generation_type":  f"aspect_{pattern}",
        "seed_genre":       genre,
        "seed_archetype":   archetype,
    }


def build_hard_negative_triple(genre: str, archetype: str,
                                model_name: str) -> Optional[dict]:
    """
    Type 3: both candidates are surface-similar to anchor
    but one has a genuinely different plot structure.
    """
    anchor = _clean(generate(
        SEED_PROMPT.format(genre=genre, archetype=archetype)))
    if not _is_valid(anchor):
        return None

    # hard_similar: same genre/setting/character type but different plot
    hard_similar = _clean(
        generate(HARD_NEGATIVE_SIMILAR_PROMPT.format(anchor=anchor)))
    # hard_distant: completely different
    hard_distant = _clean(
        generate(HARD_NEGATIVE_DISTANT_PROMPT.format(anchor=anchor)))

    if not _is_valid(hard_similar) or not _is_valid(hard_distant):
        return None

    # hard_similar is NOT closer to anchor (different plot structure)
    # hard_distant is even further
    # → Neither is "clearly" closer; this is genuinely ambiguous
    # We label hard_distant as "not closer" since it's more clearly different
    # → text_a = hard_distant, text_b = hard_similar, text_a_is_closer = False
    if random.random() < 0.5:
        text_a, text_b, closer = hard_distant, hard_similar, False
    else:
        text_a, text_b, closer = hard_similar, hard_distant, True

    return {
        "model":            model_name,
        "anchor_text":      anchor,
        "text_a":           text_a,
        "text_b":           text_b,
        "text_a_is_closer": closer,
        "generation_type":  "hard_negative",
        "seed_genre":       genre,
        "seed_archetype":   archetype,
    }


# CRASH-SAFE CACHE

def load_cache(path: str) -> set:
    """Return set of already-generated (genre, archetype, type) keys."""
    if not Path(path).exists():
        return set()
    done = set()
    with open(path, encoding="utf-8") as f:
        for line in f:
            try:
                obj = json.loads(line)
                done.add((obj["seed_genre"], obj["seed_archetype"],
                           obj["generation_type"]))
            except Exception:
                pass
    return done


def append_result(path: str, result: dict):
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(result, ensure_ascii=False) + "\n")


# GENERATION PLAN BUILDER

def build_plan(rng: random.Random) -> list:
    """
    Build a randomised list of (genre, archetype, triple_type) tasks.
    Balanced across types, genres, archetypes.
    """
    aspect_patterns = [
        "same_coa_diff_outcome", "diff_coa_same_outcome",
        "same_coa_same_outcome", "diff_coa_diff_outcome",
    ]
    n_per_aspect = N_ASPECT // len(aspect_patterns)

    tasks = []

    # Standard
    for _ in range(N_STANDARD):
        tasks.append({
            "genre":     rng.choice(GENRES),
            "archetype": rng.choice(ARCHETYPES),
            "type":      "standard",
        })

    # Aspect-controlled
    for pat in aspect_patterns:
        for _ in range(n_per_aspect):
            tasks.append({
                "genre":     rng.choice(GENRES),
                "archetype": rng.choice(ARCHETYPES),
                "type":      f"aspect_{pat}",
            })

    # Hard negatives
    for _ in range(N_HARD_NEG):
        tasks.append({
            "genre":     rng.choice(GENRES),
            "archetype": rng.choice(ARCHETYPES),
            "type":      "hard_negative",
        })

    rng.shuffle(tasks)
    return tasks


# MAIN

def main():
    # Set model name based on backend
    if BACKEND == "ollama":
        model_tag = OLLAMA_MODEL
        print(f"Backend: Ollama ({OLLAMA_MODEL})")
        
        # Check if Ollama is available
        if not _ollama_available():
            print("ERROR: Ollama is not available. Please ensure Ollama is running.")
            print("Run: ollama serve && ollama pull llama3.1:8b")
            sys.exit(1)
    else:
        model_tag = API_MODEL
        print(f"Backend: Anthropic API ({API_MODEL})")
        
        # Check if API key is available
        if not _anthropic_available():
            print("ERROR: ANTHROPIC_API_KEY not set.")
            sys.exit(1)

    print(f"Output: {OUTPUT_PATH}")
    print(f"Target: {N_STANDARD} standard + "
          f"{N_ASPECT} aspect-controlled + "
          f"{N_HARD_NEG} hard negatives = "
          f"{N_STANDARD + N_ASPECT + N_HARD_NEG} total")
    print()

    rng = random.Random(SEED)
    plan = build_plan(rng)

    # Load crash-recovery cache (the output file acts as its own cache)
    done_keys = load_cache(OUTPUT_PATH)
    print(f"Already generated: {len(done_keys)} triples (resuming)")

    # Stats tracking
    counts   = {"standard": 0, "hard_negative": 0}
    for p in ["same_coa_diff_outcome", "diff_coa_same_outcome",
               "same_coa_same_outcome", "diff_coa_diff_outcome"]:
        counts[f"aspect_{p}"] = 0
    errors   = 0
    start    = time.time()

    for i, task in enumerate(plan):
        triple_type = task["type"]
        genre       = task["genre"]
        archetype   = task["archetype"]

        cache_key = (genre, archetype, triple_type)
        if cache_key in done_keys:
            continue

        try:
            if triple_type == "standard":
                result = build_standard_triple(
                    genre, archetype, model_tag)
            elif triple_type.startswith("aspect_"):
                pattern = triple_type[len("aspect_"):]
                result  = build_aspect_triple(
                    genre, archetype, pattern, model_tag)
            else:  # hard_negative
                result  = build_hard_negative_triple(
                    genre, archetype, model_tag)

            if result is None:
                errors += 1
                print(f"  [skip] {triple_type} — generation too short")
                continue

            append_result(OUTPUT_PATH, result)
            done_keys.add(cache_key)
            counts[triple_type] = counts.get(triple_type, 0) + 1

        except Exception as e:
            errors += 1
            print(f"  [ERROR] {triple_type} ({genre}/{archetype}): {e}")
            continue

        total_done = sum(counts.values())
        if total_done % 50 == 0 and total_done > 0:
            elapsed   = time.time() - start
            rate      = total_done / elapsed
            remaining = (len(plan) - i) / rate / 60
            print(f"  [{total_done}/{len(plan)}] "
                  f"{rate:.2f} triples/s  ETA {remaining:.1f} min  "
                  f"errors={errors}")
            print(f"  Counts: {counts}")

    # Final summary
    elapsed   = time.time() - start
    total     = sum(counts.values())
    print()
    print("=" * 55)
    print(f"Generation complete in {elapsed/60:.1f} min")
    print(f"Total triples written: {total}")
    print(f"Errors / skipped    : {errors}")
    print(f"Breakdown: {counts}")
    print(f"Output file: {OUTPUT_PATH}")

    # Validate balance
    all_results = []
    with open(OUTPUT_PATH, encoding="utf-8") as f:
        for line in f:
            try:
                all_results.append(json.loads(line))
            except Exception:
                pass

    n_closer = sum(1 for r in all_results if r["text_a_is_closer"])
    n_total  = len(all_results)
    print(f"\nLabel balance: text_a_is_closer=True: "
          f"{n_closer}/{n_total} ({n_closer/n_total*100:.1f}%)")
    if not 0.40 <= n_closer / n_total <= 0.60:
        print("WARNING: Label imbalance detected. Check positional bias in prompts.")
    else:
        print("Label balance OK")
    print("=" * 55)


if __name__ == "__main__":
    main()

Delete endings like 

* conflict resolved.
* conflict unresolved.
* conflict partially resolved.
* partial resolution. 

from the outcomes field of the extracted aspects

In [ ]:
import json
import re
from pathlib import Path


# Only match labels at the END of the outcomes string
END_LABEL_PATTERNS = [
    (re.compile(r"(?:\s*[.;,:-]?\s*)conflict\s+unresolved\s*[.;,:-]?\s*$", re.I), "unresolved"),
    (re.compile(r"(?:\s*[.;,:-]?\s*)conflict\s+partially\s+resolved\s*[.;,:-]?\s*$", re.I), "partial"),
    (re.compile(r"(?:\s*[.;,:-]?\s*)partial\s+resolution\s*[.;,:-]?\s*$", re.I), "partial"),
    (re.compile(r"(?:\s*[.;,:-]?\s*)conflict\s+resolved\s*[.;,:-]?\s*$", re.I), "resolved"),
]


def extract_end_resolution_status(outcomes: str):
    if not outcomes:
        return outcomes, None

    text = outcomes.strip()

    for pattern, label in END_LABEL_PATTERNS:
        match = pattern.search(text)
        if match:
            cleaned = text[:match.start()].rstrip(" \t\n\r.;,:-")
            return cleaned, label

    return text, None


def process_payload(payload: dict) -> dict:
    outcomes = payload.get("outcomes", "")
    cleaned_outcomes, status = extract_end_resolution_status(outcomes)

    payload["outcomes"] = cleaned_outcomes
    payload["resolution_status"] = status
    return payload


def process_data(data):
    if isinstance(data, dict):
        out = {}
        for key, value in data.items():
            if isinstance(value, dict):
                out[key] = process_payload(value.copy())
            else:
                out[key] = value
        return out

    if isinstance(data, list):
        out = []
        for item in data:
            if isinstance(item, dict):
                out.append(process_payload(item.copy()))
            else:
                out.append(item)
        return out

    raise ValueError("Unsupported input format. Expected JSON object or list.")


def load_json_or_jsonl(path: Path):
    text = path.read_text(encoding="utf-8").strip()
    if not text:
        raise ValueError("Input file is empty.")

    try:
        return json.loads(text), "json"
    except json.JSONDecodeError:
        pass

    rows = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows, "jsonl"


def save_json_or_jsonl(data, path: Path, fmt: str):
    if fmt == "json":
        path.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")
    elif fmt == "jsonl":
        with path.open("w", encoding="utf-8") as f:
            for row in data:
                f.write(json.dumps(row, ensure_ascii=False) + "\n")
    else:
        raise ValueError(f"Unknown output format: {fmt}")


def main():
    input_path = Path("/kaggle/input/datasets/similarity/extracted_aspects_dev_synth_extra_synth.json")
    output_path = Path("/kaggle/working/clean_extracted_aspects.json")

    data, fmt = load_json_or_jsonl(input_path)
    cleaned = process_data(data)
    save_json_or_jsonl(cleaned, output_path, fmt)

    print(f"Saved cleaned file to: {output_path}")


if __name__ == "__main__":
    main()